# Cas d'usage 01 — Résiliation client SaaS (churn)
### Certification « Concevoir et implémenter une solution d'intelligence artificielle »

**Candidat :** [Prénom NOM]
**Date :** [JJ/MM/AAAA]
**Version du notebook :** v1.0
**Environnement d'exécution :** Python [X.X], JupyterLab, dépendances listées en Annexe (section 15)

---

**Modalités d'évaluation.** Ce notebook constitue la *production écrite* (production sur jeux de données
+ journal de bord). Il est complété par une soutenance orale de 30 minutes suivie de 30 minutes d'échanges
avec le jury. Les deux modalités pèsent à parité dans l'évaluation du cas d'usage.

**Couverture des compétences.** Chaque section porte le marqueur `[Cn]` de la compétence qu'elle démontre.
La table de correspondance complète entre les critères du référentiel et leur emplacement dans ce document
figure en **Annexe B** (section 15).

---


## 1. Résumé exécutif

**Problème.** Prédire la résiliation (churn) d'un compte client B2B SaaS à l'échéance de son contrat, à partir de
données d'usage, de facturation, de support et de caractéristiques contractuelles.

**Objectif métier.** Fournir aux équipes Customer Success un score de risque de churn par compte, exploitable en
priorisation d'actions de rétention — pas une décision automatisée.

**Approche.** Apprentissage supervisé :
- Cible principale : `churn` (classification binaire).
- Cible secondaire : `valeur_vie_client_eur` (régression), traitée séparément, jamais utilisée comme feature du
  modèle de churn.

**Résultats clés.** *(à compléter après exécution : AUC, PR-AUC, seuil retenu, variables les plus influentes,
nombre de comptes à risque détectés).*

**Limites.** *(à compléter : incertitude sur les données synthétiques, biais potentiels, variables leurres,
hypothèses de coût métier pour le seuil).*

> **Choix rédactionnel :** le résumé exécutif est rédigé en dernier (une fois les résultats connus) mais placé en
> tête du notebook, conformément au plan imposé — c'est la section lue en premier par le jury.


## 2. Cadrage métier et cas d'usage — journal de bord [C1]

**Contexte.** Éditeur SaaS B2B, revenus récurrents (MRR), churn = enjeu de rentabilité et de croissance.

**Cas d'usage IA.** Score de risque de churn par compte, calculé avant l'échéance contractuelle, pour prioriser les
actions de rétention des équipes Customer Success.

**Ce que le modèle n'est pas.** Pas un outil de décision automatique de résiliation/rétention ; pas un substitut au
jugement humain du CSM (Customer Success Manager).

**Utilisateurs finaux.** Équipes Customer Success, éventuellement Sales Ops pour le reporting.

**Journal de bord [C1] :**
- Décision : cibler le churn *contractuel à l'échéance* (et non un churn "à tout moment"), car c'est la fenêtre
  d'action pertinente pour le Customer Success.
- Décision : traiter `valeur_vie_client_eur` comme éclairage métier complémentaire (priorisation des comptes à
  forte valeur), jamais comme feature d'entrée du modèle churn (cf. section 3.2 de l'énoncé).

**Hypothèses et contraintes.**

| Nature | Énoncé |
|---|---|
| Hypothèse | Les données fournies sont représentatives du portefeuille réel de comptes |
| Hypothèse | Une action de rétention menée à temps a un effet non nul sur la décision du client |
| Hypothèse | Le coût d'un compte perdu excède le coût d'une action de rétention inutile |
| Contrainte | Décision humaine obligatoire — aucun déclenchement automatique |
| Contrainte | Fenêtre d'action : quelques semaines avant l'échéance contractuelle |
| Contrainte | Score restitué dans le CRM existant, sans outil supplémentaire pour les CSM |
| Contrainte | Explicabilité exigée : un CSM doit comprendre pourquoi un compte est signalé |

**Critères de réussite.**

| Niveau | Critère |
|---|---|
| Technique | Modèle supérieur à la baseline sur le PR-AUC, performance stable en validation croisée |
| Opérationnel | Volume de comptes signalés compatible avec la capacité de traitement des CSM |
| Métier | Taux de rétention des comptes signalés supérieur à celui des comptes non signalés |
| Conformité | Aucune variable sensible ou en fuite dans le modèle, traçabilité des décisions |

**Cadre de suivi défini dès le cadrage.** La pertinence des indicateurs de performance et de leurs seuils est
révisée **trimestriellement**, au même rythme que le réentraînement (mise en œuvre en section 13). Fixer cette
périodicité dès la phase de cadrage évite qu'un dispositif de suivi conçu à la mise en service reste inchangé
alors que le contexte métier a évolué.

**Journal de bord [C1] — complément :**
- Décision : retenir comme critère de réussite métier l'écart de rétention entre comptes signalés et non
  signalés, et non la performance statistique. Motif : un modèle excellent dont les alertes ne changent rien
  au comportement des équipes ne crée aucune valeur.
- Décision : inscrire la contrainte de volume dès le cadrage. Motif : le seuil de décision doit produire un
  nombre de comptes traitable par les CSM — une contrainte opérationnelle, pas seulement statistique.

### Ordres de grandeur métier constatés sur les données

Ces chiffres, établis avant toute modélisation, conditionnent le cadrage de la solution.

| Constat | Valeur |
|---|---|
| Comptes uniques | 5 000 (5 035 lignes, 35 doublons stricts) |
| Taux de churn | 28 % — 1 400 comptes |
| MRR mensuel total du portefeuille | ≈ 17,2 M€ |
| MRR porté par les comptes qui résilient | ≈ 4,0 M€, soit 23 % du portefeuille |
| Concentration : 1 % des comptes | 17 % du MRR total |
| Concentration : 10 % des comptes | 68 % du MRR total |
| Concentration : 10 % des churners | 72 % du MRR perdu |

**Conséquence de cadrage.** La valeur est très concentrée : traiter les 10 % de churners les plus importants
couvrirait près des trois quarts du revenu à risque. Le problème n'est donc pas de détecter *tous* les
départs, mais de détecter ceux qui comptent. Cela oriente la solution vers une **priorisation par valeur**
plutôt que vers une maximisation du rappel global (voir section 9).

**Périmètre temporel — hypothèse de lecture des données.** Le jeu de données est un **instantané** : chaque
ligne décrit un compte et l'issue observée à son échéance, sans horodatage de la décision. Les 1 400
résiliations ne sont donc pas un flux mensuel mais un cumul sur la période couverte. Hypothèse retenue pour
raisonner en volumétrie : ces résiliations se répartissent sur environ douze mois, soit de l'ordre de
**115 à 120 échéances à risque par mois**. Cette hypothèse est explicitée car elle conditionne le
dimensionnement de la capacité ci-dessous ; elle serait à remplacer par le calendrier réel des échéances en
situation opérationnelle.

**Articulation avec la fenêtre de décision.** Le scoring est mensuel, mais la décision se prend à l'échéance
contractuelle de chaque compte. Le lot mensuel n'a donc pas vocation à traiter tout le portefeuille : il
cible les comptes dont l'échéance approche, croisés avec leur score de risque (cf. section 11).

**Capacité de traitement — hypothèse structurante.** Le nombre de comptes actionnables par mois est une
contrainte dure, indépendante du modèle. Hypothèse retenue : **≈ 140 comptes par mois**, soit l'ordre de
grandeur permettant de couvrir les 10 % de churners les plus valorisés. Cette valeur est à confirmer auprès
du commanditaire ; elle détermine directement le point de fonctionnement du modèle.

**Journal de bord [C1] — complément :**
- Décision : établir les ordres de grandeur métier avant la modélisation, et non après. Motif : ils
  déterminent le type de solution à construire — ici, un outil de priorisation et non un détecteur exhaustif.
- Décision : traiter la capacité CSM comme une contrainte de cadrage et non comme un paramètre d'ajustement
  a posteriori. Motif : un modèle qui signale plus de comptes que l'équipe ne peut en traiter ne produit
  aucune action supplémentaire.

### Cycle de vie du projet et positionnement des sections

La démarche suit un cycle de vie itératif, et non un enchaînement linéaire. Le plan du notebook étant imposé
et séquentiel, cette table établit la correspondance entre les deux lectures.

```mermaid
flowchart LR
    A[Cadrage] --> B[Données]
    B --> C[Features]
    C --> D[Modélisation]
    D --> E[Évaluation]
    E --> F[Packaging]
    F --> G[Industrialisation]
    G --> H[Monitoring]
    H -. dérive / nouvelle donnée .-> B
    E -. score insuffisant .-> C
    D -. signal faible .-> A
```

*(Le diagramme est reproduit par la figure générée ci-dessous, Jupyter ne rendant pas nativement la syntaxe
Mermaid.)*

| Étape du cycle | Sections du notebook |
|---|---|
| **A — Cadrage** | 2 (besoin, contraintes, critères) · 4 (cadre éthique et réglementaire) |
| **B — Données** | 3 (sources, gouvernance, cycle de vie) · 5 (chargement, contrôles) |
| **C — Features** | 6 (analyse exploratoire) · 7 (nettoyage, transformations, variables construites) |
| **D — Modélisation** | 8 (familles, baseline, démarche) · 9 (entraînement, validation) |
| **E — Évaluation** | 9 (validation avant industrialisation) · 12 (performance et impacts en exploitation) |
| **F — Packaging** | 10 (sérialisation, versioning, fiche modèle) |
| **G — Industrialisation** | 10 (chaîne CI/CD, intégration) · 11 (architecture cible) |
| **H — Monitoring** | 13 (dérive, seuils, réentraînement) |

**Écart assumé entre le cycle et le plan imposé.** Le règlement place l'implémentation (10) et l'architecture
(11) *avant* la mesure de performance (12), alors que le cycle évalue avant de packager. Il n'y a pas de
contradiction : l'évaluation se déroule en deux temps distincts. La **validation avant mise en service**
(section 9) décide si le modèle mérite d'être industrialisé ; la **mesure en exploitation** (section 12)
vérifie ce qu'il produit réellement une fois déployé. Le plan du notebook suit l'ordre du règlement ; la
démarche réelle suit le cycle.

**Les trois boucles de rétroaction** sont documentées là où elles se sont produites : la boucle *modélisation
→ cadrage* en section 8, la boucle *évaluation → features* en section 9, la boucle *monitoring → données* en
section 13. Le journal des itérations en annexe (section 15) les récapitule.


In [ ]:
# Figure : cycle de vie du projet (reproduction du diagramme Mermaid ci-dessus)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

etapes = ["Cadrage", "Données", "Features", "Modélisation",
          "Évaluation", "Packaging", "Industrialisation", "Monitoring"]
sections = ["2, 4", "3, 5", "6, 7", "8, 9", "9, 12", "10", "10, 11", "13"]

fig, ax = plt.subplots(figsize=(15, 4.5))
largeur, hauteur = 1.55, 0.7

for i, (etape, sec) in enumerate(zip(etapes, sections)):
    x = i * 1.85
    ax.add_patch(FancyBboxPatch((x, 0), largeur, hauteur,
                                boxstyle="round,pad=0.04", linewidth=1.2,
                                edgecolor="#333333", facecolor="#eef2f7"))
    ax.text(x + largeur / 2, 0.44, etape, ha="center", va="center",
            fontsize=10, fontweight="bold")
    ax.text(x + largeur / 2, 0.20, f"§ {sec}", ha="center", va="center",
            fontsize=8.5, color="#555555")
    if i < len(etapes) - 1:
        ax.add_patch(FancyArrowPatch((x + largeur, hauteur / 2), (x + 1.85, hauteur / 2),
                                     arrowstyle="-|>", mutation_scale=13,
                                     linewidth=1.2, color="#333333"))

# Boucles de retroaction
boucles = [
    (3, 0, 1.30, "signal faible", "#b03030"),   # Modelisation -> Cadrage
    (4, 2, 1.05, "score insuffisant", "#b03030"),  # Evaluation -> Features
]
for src, dst, haut, libelle, couleur in boucles:
    xs = src * 1.85 + largeur / 2
    xd = dst * 1.85 + largeur / 2
    ax.add_patch(FancyArrowPatch((xs, hauteur), (xd, hauteur),
                                 connectionstyle=f"arc3,rad=0.28",
                                 arrowstyle="-|>", mutation_scale=12, linewidth=1.1,
                                 linestyle="--", color=couleur))
    ax.text((xs + xd) / 2, haut, libelle, ha="center", va="bottom",
            fontsize=8.5, color=couleur, style="italic")

# Monitoring -> Donnees (par le bas)
xs, xd = 7 * 1.85 + largeur / 2, 1 * 1.85 + largeur / 2
ax.add_patch(FancyArrowPatch((xs, 0), (xd, 0),
                             connectionstyle="arc3,rad=-0.22",
                             arrowstyle="-|>", mutation_scale=12, linewidth=1.1,
                             linestyle="--", color="#2a6f9e"))
ax.text((xs + xd) / 2, -0.55, "dérive / nouvelle donnée", ha="center", va="top",
        fontsize=8.5, color="#2a6f9e", style="italic")

ax.set_xlim(-0.4, len(etapes) * 1.85)
ax.set_ylim(-0.95, 1.7)
ax.axis("off")
ax.set_title("Cycle de vie du projet et correspondance avec les sections du notebook",
             fontsize=11, pad=14)
plt.tight_layout()
plt.show()


## 3. Données : disponibilité, gouvernance et alternatives — journal de bord [C1]

**Sources fournies :**
| Fichier | Contenu | Usage prévu |
|---|---|---|
| `churn_saas_complet.csv` | ~5000 lignes, 29 colonnes | Entraînement / évaluation |
| `churn_saas_echantillon.csv` | 50 lignes | Contrôle rapide, tests unitaires de pipeline |
| `catalogue_plans.csv` | Référentiel des plans (prix/siège, SLA, stockage, support dédié) | Jointure enrichissante sur `plan` |

**Gouvernance / propriété des données.** Données internes (usage, facturation, support) : propriété de l'éditeur
SaaS. Champ `commentaire_csm` = texte libre potentiellement sensible (mentions nominatives, jugements qualitatifs)
→ à traiter avec prudence (cf. section 4, éthique/RGPD).

**Alternatives envisageables (non retenues ici, à mentionner pour le jury).** Données de télémétrie produit plus
fines (logs par fonctionnalité), données NPS, historique des tickets support détaillé (contenu texte) — non
disponibles dans ce jeu de données, pistes d'enrichissement futures (cf. section 13).

**Journal de bord [C1] :**
- Décision : jointure `churn_saas_complet` ↔ `catalogue_plans` sur la clé `plan`, en normalisant la casse
  (`STARTER` vs `Starter`) avant la jointure.

**Disponibilité et accès — vérifications effectuées.**

| Point contrôlé | Constat |
|---|---|
| Existence des fichiers | Les trois fichiers sont présents et lisibles |
| Encodage | UTF-8 avec BOM sur certains fichiers — à traiter au chargement (section 5) |
| Complétude | Valeurs manquantes présentes sur plusieurs colonnes — quantifiées en section 5 |
| Cohérence de la clé de jointure | Casse hétérogène sur `plan` — normalisation requise avant jointure |
| Doublons | Présence de lignes dupliquées — traitées en section 7 |
| Droits d'usage | Données internes à l'éditeur, usage interne d'aide à la décision |

**Solutions alternatives en cas d'indisponibilité des données.** Ces scénarios sont anticipés car ils
conditionnent la viabilité de la solution en production, pas seulement dans ce notebook.

| Indisponibilité | Alternative retenue |
|---|---|
| Perte d'accès au référentiel des plans | Dégradation maîtrisée : le modèle tourne sans les variables issues de la jointure, au prix d'une performance moindre, à mesurer |
| Variable d'usage manquante sur une période | Imputation par la dernière valeur connue du compte, avec indicateur de manquant conservé comme signal |
| Indisponibilité complète de l'historique d'usage | Repli sur un modèle restreint aux variables contractuelles et de facturation, moins performant mais opérationnel |
| Volumétrie insuffisante pour réentraîner | Maintien du modèle en place, alerte au commanditaire, réentraînement différé |

**Modèle de stockage retenu.**

| Option | Évaluation | Décision |
|---|---|---|
| Base de données relationnelle | Données tabulaires, schéma stable, jointures nécessaires, volumétrie modeste | **Retenue** |
| Stockage objet (fichiers) | Adapté aux instantanés d'entraînement et à l'archivage versionné | **Retenue en complément** |
| Base orientée documents | Aucun besoin de schéma variable ni de structure imbriquée | Écartée |
| Entrepôt distribué | Volumétrie très inférieure au seuil de pertinence | Écarté — sur-dimensionnement |

Le couple relationnel + stockage objet répond aux deux besoins distincts : interrogation courante des données
métier d'un côté, conservation d'instantanés d'entraînement immuables de l'autre — ces derniers étant la
condition de la reproductibilité exigée en section 10.

**Cycle de vie du jeu de données.**

| Étape | Contenu | Durée / rythme | Responsable |
|---|---|---|---|
| Collecte | Extraction depuis les systèmes d'usage, facturation et support | Mensuelle | Équipe Data |
| Préparation | Nettoyage, jointure, feature engineering (section 7) | À chaque extraction | Équipe Data |
| Instantané d'entraînement | Figé et versionné à chaque réentraînement | Trimestriel | Équipe Data |
| Exploitation | Scoring, écriture des résultats dans le CRM | Mensuelle | Équipe Data / Ops CRM |
| Conservation des scores | Historique nécessaire au suivi de dérive | À arbitrer avec le DPO | DPO |
| Archivage | Instantanés d'entraînement conservés pour audit et retour arrière | Durée alignée sur les versions de modèle actives | Équipe Data |
| Purge | Suppression des données et scores au-delà de la durée retenue | Automatisée | Équipe Data / DPO |

**Soumission aux parties prenantes.** Ce cycle de vie est présenté au DPO pour validation des durées de
conservation et de la base légale, au commanditaire pour l'arbitrage sur la profondeur d'historique, et à
l'équipe infrastructure pour la faisabilité de l'archivage et de la purge automatisée. La durée de
conservation des scores est le point ouvert : le suivi de dérive demande de la profondeur, la minimisation
des données demande l'inverse.

**Journal de bord [C1] / [C3] — complément :**
- Décision : documenter des scénarios de dégradation plutôt que de supposer les données toujours
  disponibles. Motif : un pipeline qui échoue dès qu'une source manque n'est pas exploitable en production.
- Décision : conserver l'indicateur de valeur manquante comme variable à part entière lorsqu'une imputation
  est faite. Motif : l'absence d'une donnée d'usage peut être elle-même un signal de désengagement.
- Décision : laisser ouverte la durée de conservation des scores plutôt que de la fixer unilatéralement.
  Motif : l'arbitrage entre suivi de dérive et minimisation des données relève du DPO et du commanditaire.


In [ ]:
import pandas as pd
import numpy as np

PATH_COMPLET = "churn_saas_complet.csv"
PATH_ECH = "churn_saas_echantillon.csv"
PATH_CATALOGUE = "catalogue_plans.csv"

# Raw loading (no advanced parsing at this stage - see section 5)
df_raw = pd.read_csv(PATH_COMPLET)
df_catalogue = pd.read_csv(PATH_CATALOGUE)

print(df_raw.shape, df_catalogue.shape)
df_catalogue.head()


## 4. Enjeux éthiques, sociétaux et conformité — journal de bord [C2]

**Risques de biais identifiés.**
- `secteur`, `pays`, `taille_entreprise` : variables catégorielles pouvant agir comme proxys de discrimination
  indirecte (ex. sous-priorisation systématique d'un secteur ou d'un pays). À surveiller en post-hoc (taux de
  score de risque par groupe).
- `groupe_experimentation` (bucket A/B) : ne doit pas être utilisé comme feature prédictive (artefact
  expérimental, pas un signal métier stable) — risque de sur-apprentissage d'un effet de test interne.

**RGPD / données personnelles.**
- `client_id` : identifiant pseudonymisé mais reliable à une entreprise cliente — à exclure du modèle
  (aucune valeur prédictive, uniquement identifiant).
- `commentaire_csm` : champ libre pouvant contenir des données personnelles ou des jugements subjectifs sur des
  personnes physiques → exclusion du modèle, traitement séparé si exploité un jour (NLP) avec base légale à
  vérifier.

**Conséquences métier des erreurs.**
- Faux négatif (client à risque non détecté) : perte du client, perte de MRR, aucune action corrective déclenchée.
- Faux positif (client sain classé à risque) : coût d'une action de rétention "inutile" (remise, appel CSM),
  de l'ordre de 135 € (temps CSM chargé et geste commercial pondéré).
- **L'asymétrie existe mais n'est pas uniforme.** Le coût d'un faux négatif étant proportionnel à la valeur du
  compte, le rapport C_FN/C_FP varie de **1,5 pour un compte du premier décile à plus de 300 pour un compte du
  dernier décile**. Affirmer que le faux négatif coûte « structurellement » plus cher serait donc faux pour une
  partie du portefeuille : sur les plus petits comptes, les deux erreurs ont un coût comparable.
- → la conséquence n'est pas un seuil global privilégiant le rappel, mais une **règle de décision propre à
  chaque compte**, détaillée en section 9. Le rappel reste prioritaire sur la précision pour les comptes à
  forte valeur, qui concentrent l'essentiel du revenu à risque.

**Variables à écarter pour raisons éthiques/anti-fuite (détail méthodologique en section 7) :**
`client_id` (identifiant), `sante_compte_fin_periode` (fuite temporelle), `commentaire_csm` (texte libre non
structuré / sensible), `groupe_experimentation` (artefact expérimental).

**Journal de bord [C2] :**
- Décision : ne pas utiliser le score de churn comme déclencheur automatique d'action commerciale défavorable
  (ex. augmentation tarifaire) — usage strictement en aide à la décision humaine.

**Cadres de référence applicables.**
- **RGPD** (règlement UE 2016/679) : base légale du traitement (intérêt légitime de l'éditeur pour la
  rétention client), minimisation des données, durée de conservation, droit d'opposition. `commentaire_csm`
  et `client_id` sont traités au titre de ce cadre.
- **Règlement européen sur l'IA (AI Act, UE 2024/1689)** : le système relève de la catégorie « risque
  minimal ». Il ne s'agit ni d'un scoring de crédit, ni d'un usage RH, ni d'une notation sociale — les
  obligations renforcées ne s'appliquent donc pas. L'obligation retenue volontairement est celle de
  **transparence** : les CSM savent qu'un score algorithmique alimente leur priorisation.
- **Lignes directrices européennes pour une IA digne de confiance** (groupe d'experts de haut niveau,
  Commission européenne) : les principes de supervision humaine, de robustesse technique et
  d'explicabilité structurent les choix de ce notebook — la supervision humaine est garantie par le refus
  de toute décision automatisée (voir journal de bord).
- **Recommandations de la CNIL sur l'IA** : analyse d'impact non obligatoire ici (pas de données sensibles
  au sens de l'article 9, pas de décision produisant des effets juridiques), mais le registre des
  traitements doit être mis à jour.

**Dilemmes éthiques identifiés.**

1. **Cibler un client à risque peut accélérer son départ.** Une sollicitation de rétention mal calibrée
   (remise défensive, appel inhabituel) signale au client qu'il est considéré comme partant et peut
   déclencher une réflexion qu'il n'avait pas engagée. *Position retenue :* le score déclenche une action
   de valeur (accompagnement, formation, revue d'usage), jamais un geste commercial défensif immédiat.
   Le CSM reste libre de ne pas agir.
2. **Concentrer l'effort sur les comptes à forte valeur crée une inégalité de traitement.** Prioriser par
   `MRR × probabilité de churn` revient à délaisser structurellement les petits comptes, qui sont aussi
   ceux dont le départ est le moins coûteux. *Position retenue :* la priorisation par valeur est assumée
   comme une décision commerciale, mais un volume minimal d'actions est réservé aux petits comptes afin
   d'éviter un effet d'abandon systématique. L'arbitrage appartient au commanditaire, pas au modèle.

**Acteurs informés des risques identifiés.**

| Acteur | Risque porté à sa connaissance | Attendu |
|---|---|---|
| Commanditaire (direction Customer Success) | Dilemmes 1 et 2, limites du modèle | Arbitrage sur la politique de priorisation |
| DPO / juriste | Base légale, `commentaire_csm`, durée de conservation | Validation de la conformité RGPD, mise à jour du registre |
| Équipes CSM | Nature probabiliste du score, absence d'automatisation | Compréhension de l'outil, remontée des faux positifs |
| Équipe technique | Variables exclues et motifs | Respect des exclusions en production |

**Journal de bord [C2] — complément :**
- Décision : documenter les deux dilemmes plutôt que de les trancher unilatéralement. Le choix de politique
  de priorisation relève du commanditaire ; le rôle du concepteur est de rendre l'arbitrage visible.
- Décision : retenir l'obligation de transparence auprès des CSM alors que l'AI Act ne l'impose pas pour un
  système à risque minimal. Motif : la supervision humaine n'est effective que si l'utilisateur sait qu'un
  algorithme intervient.

### Conséquence éthique de la priorisation par valeur

La section 9 retient un classement des comptes par **valeur espérée** (probabilité × valeur du compte) plutôt
que par probabilité seule. Ce choix est économiquement fondé, mais il a une conséquence qu'il serait
malhonnête de passer sous silence : **il rend les petits comptes structurellement moins prioritaires**.

L'ampleur de l'effet est mesurable. Le seuil de probabilité économiquement rationnel varie de 0,40 pour un
compte du premier décile de valeur à 0,003 pour un compte du dernier décile — un facteur 130. À probabilité
de départ égale, un petit compte ne sera donc pratiquement jamais contacté.

**Ce n'est pas une discrimination au sens juridique** : la valeur d'un compte est un critère commercial
légitime, sans lien avec une caractéristique protégée. Mais c'est un arbitrage qui doit être posé
explicitement, pour trois raisons :

1. Il peut produire un effet d'auto-réalisation — les petits comptes délaissés résilient davantage, ce qui
   renforce leur faible valeur estimée au cycle suivant.
2. Il peut entrer en tension avec un engagement commercial d'égalité de traitement entre clients.
3. Il concentre le risque : le portefeuille de petits comptes se dégrade sans que l'indicateur global de
   rétention pondérée par la valeur ne le signale.

**Mesures de limitation retenues.**

| Mesure | Modalité |
|---|---|
| Quota réservé | Une part du volume mensuel d'actions est allouée aux comptes hors priorisation par valeur |
| Suivi séparé | Le taux de rétention est suivi par segment de taille, pas seulement en agrégat |
| Traitement différencié | Les petits comptes relèvent d'actions à faible coût unitaire (campagne, contenu d'accompagnement) plutôt que d'un contact individuel |
| Arbitrage explicite | La pondération valeur/volume est validée par le commanditaire, pas fixée par l'équipe technique |

**Journal de bord [C2] — complément :**
- Décision : quantifier l'effet de la priorisation par valeur (facteur 130 entre déciles extrêmes) plutôt que
  de la mentionner qualitativement. Motif : un arbitrage n'est discutable par le commanditaire que s'il est
  chiffré.
- Décision : ne pas corriger l'effet dans le modèle, mais le compenser dans la politique d'action. Motif :
  altérer le score pour le rendre « équitable » masquerait l'arbitrage au lieu de l'exposer ; la correction
  relève de la décision métier, pas de l'algorithme.


In [ ]:
# Section 4 - no code needed (ethical analysis), cell reserved if needed


## 5. Chargement et compréhension des données [C3]

**Constats bruts observés sur `churn_saas_complet.csv` :**
- BOM UTF-8 en tête de fichier (`ï»¿` avant `client_id`).
- Nombres décimaux stockés en texte avec virgule et guillemets : `"0,0"`, `"33,3"`, `"40,0"` (colonnes
  `taux_adoption_pct`, `heures_usage_30j`, `delai_reponse_support_h`...).
- Dates en formats hétérogènes dans `date_souscription` (`31/01/2024`, `2024-02-06`, formats textuels).
- Casse hétérogène sur les catégorielles (`STARTER` vs `Pro`, `SANTÉ` vs `Santé`).
- Doublons potentiels (à quantifier).
- Valeurs manquantes multiples (`commentaire_csm` très souvent vide, autres colonnes à auditer).

**Journal de bord [C3] :**
- Décision : dresser un état des lieux systématique (types, % manquants, doublons, valeurs aberrantes) avant toute
  transformation, pour documenter objectivement le "avant / après" nettoyage (traçabilité exigée par la
  certification).

**Contrôles de cohérence attendus — repères chiffrés.** Les valeurs ci-dessous servent de référence : tout
écart significatif après exécution signale un problème de chargement ou de parsing.

| Contrôle | Valeur de référence |
|---|---|
| Lignes brutes | 5 035 |
| Doublons stricts | 35 → 5 000 comptes uniques |
| Taux de churn | 28,0 % |
| `commentaire_csm` manquant | 55,5 % |
| `delai_reponse_support_h` manquant | 10,0 % |
| `csat` manquant | 8,0 % |
| `heures_usage_30j` manquant | 6,0 % |
| `taux_adoption_pct`, `retards_paiement_12m`, `secteur` manquants | 5,0 % |
| `pays`, `nb_integrations` manquants | 4,0 % |
| `revenu_mensuel_recurrent_eur` manquant | 3,0 % |

Le taux de manquants est régulier — de 3 à 10 % — sur un ensemble de colonnes, ce qui suggère une absence
introduite de façon aléatoire plutôt qu'un mécanisme métier. Cette hypothèse est à vérifier en section 7
avant de choisir la stratégie d'imputation : un manquant corrélé au churn serait lui-même un signal.


In [ ]:
# Data dictionary (taken from the assignment, for documentation and consistency checks)
DATA_DICT = {
    "client_id": "identifiant unique (texte) — à exclure du modèle",
    "date_souscription": "date, formats mêlés",
    "jour_souscription": "catégoriel",
    "secteur": "catégoriel",
    "pays": "catégoriel",
    "taille_entreprise": "catégoriel (TPE/PME/ETI/GE)",
    "plan": "catégoriel — clé de jointure avec catalogue_plans",
    "anciennete_mois": "entier",
    "sieges_souscrits": "entier",
    "utilisateurs_actifs": "entier (<= sieges_souscrits)",
    "taux_adoption_pct": "décimal — stocké en texte avec virgule",
    "connexions_30j": "entier",
    "heures_usage_30j": "décimal — stocké en texte avec virgule",
    "fonctionnalites_total": "entier",
    "fonctionnalites_utilisees": "entier",
    "nb_integrations": "entier",
    "derniere_connexion_jours": "entier",
    "tickets_support_90j": "entier",
    "delai_reponse_support_h": "décimal",
    "csat": "entier (1-5)",
    "retards_paiement_12m": "entier",
    "revenu_mensuel_recurrent_eur": "décimal (MRR)",
    "couleur_theme_interface": "catégoriel — LEURRE présumé",
    "code_datacenter": "catégoriel — LEURRE présumé",
    "groupe_experimentation": "catégoriel (A/B) — artefact expérimental, exclu des features (cf. §12)",
    "commentaire_csm": "texte libre, souvent vide — à exclure",
    "sante_compte_fin_periode": "entier — PIEGE DE FUITE, calculé en FIN de période",
    "valeur_vie_client_eur": "cible secondaire (régression) — ne jamais utiliser en feature du modèle churn",
    "churn": "cible principale (classification binaire)",
}

# Current state: types, missing values, duplicates
info_types = df_raw.dtypes
pct_manquants = df_raw.isna().mean().sort_values(ascending=False) * 100
nb_doublons = df_raw.duplicated().sum()
nb_doublons_id = df_raw.duplicated(subset=["client_id"]).sum()

print("Doublons stricts (toutes colonnes) :", nb_doublons)
print("Doublons sur client_id :", nb_doublons_id)
pct_manquants[pct_manquants > 0]


## 6. Analyse exploratoire (EDA) : visualisations et interprétation — journal de bord [C3]

**Objectifs de l'EDA.**
1. Comprendre la distribution de la cible (`churn`) — vérifier le déséquilibre de classe.
2. Explorer les relations bivariées entre variables candidates et `churn` (usage, support, contrat).
3. Repérer visuellement les variables à performance suspecte (ex. `sante_compte_fin_periode`) et les leurres.
4. Vérifier la cohérence catalogue_plans ↔ churn (ex. le prix/siège influence-t-il le churn ?).

**Journal de bord [C3] :**
- Décision : produire systématiquement une lecture *univariée* (distribution) puis *bivariée* (vs `churn`) pour
  chaque variable numérique clé, plutôt qu'une matrice de corrélation seule — plus interprétable pour un public
  métier non technique lors de la soutenance.
- Décision : vérifier explicitement si `sante_compte_fin_periode` sépare quasi parfaitement `churn` = signal
  d'alerte fuite de données (à confirmer en section 7/8).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 6.1 Target distribution
df_raw["churn"].value_counts(normalize=True).plot(kind="bar", title="Répartition churn (0/1)")
plt.show()

# 6.2 Numeric variables vs churn (example to generalize)
# WARNING: at this stage some columns are still text format (decimal comma) -> need cleaning
# before a real quantitative reading (see section 7). Preliminary EDA here on already-numeric columns.
num_cols_ok = ["anciennete_mois", "sieges_souscrits", "utilisateurs_actifs",
               "connexions_30j", "derniere_connexion_jours", "tickets_support_90j",
               "csat", "retards_paiement_12m", "sante_compte_fin_periode"]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.ravel(), num_cols_ok):
    sns.boxplot(data=df_raw, x="churn", y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

# 6.3 Potential leakage alert: near-perfect separation?
sns.boxplot(data=df_raw, x="churn", y="sante_compte_fin_periode")
plt.title("ALERTE ANTI-FUITE : sante_compte_fin_periode vs churn")
plt.show()


## 7. Préparation des données (nettoyage, manquants, transformations, features) — journal de bord [C3]

**Plan de nettoyage :**
1. Suppression du BOM, normalisation de la casse des catégorielles (`.str.strip().str.lower()` ou mapping
   explicite vers un référentiel de labels).
2. Conversion des décimaux texte → float : remplacement `,` → `.`, suppression des unités/symboles (`%`, `€`) si
   présents, cast `float`.
3. Parsing multi-format de `date_souscription` (`pd.to_datetime` avec `format=None` / essai successif de formats,
   ou `dateutil`), puis dérivation de features (ex. ancienneté recalculée en jours si utile).
4. Détection et suppression des doublons (stricts + sur `client_id`, en gardant la ligne la plus complète en cas
   de conflit).
5. Traitement des valeurs manquantes : stratégie **documentée par colonne** (ex. médiane pour les variables
   d'usage numériques peu asymétriques, mode pour les catégorielles, indicateur de manquance si le manque est
   informatif, suppression de ligne uniquement si le taux de manquants est marginal).
6. Contrôles de cohérence métier (ex. `utilisateurs_actifs <= sieges_souscrits`, `fonctionnalites_utilisees <=
   fonctionnalites_total`) — traitement des incohérences résiduelles (clip ou exclusion documentée).

**Anti-fuite — exclusions actées :**
- `client_id` : identifiant, aucune valeur prédictive.
- `sante_compte_fin_periode` : **piège de fuite** — score calculé en fin de période, donc partiellement postérieur
  à la décision de résiliation ; l'utiliser reviendrait à prédire le passé avec de l'information du futur.
- `commentaire_csm`, `groupe_experimentation` : hors périmètre modèle (texte libre non structuré / artefact
  expérimental), cf. section 4.

**Variables "leurres" identifiées (conservées pour test d'importance mais faible pouvoir prédictif attendu) :**
`couleur_theme_interface`, `code_datacenter` — a priori sans lien causal avec le churn ; à confirmer
empiriquement en section 9 (importance quasi nulle attendue) plutôt qu'exclues a priori, pour illustrer la
démarche d'interprétabilité (C4/C8).

**Feature engineering (ratios métier) :**
- `taux_utilisation_sieges = utilisateurs_actifs / sieges_souscrits`
- `taux_usage_fonctionnalites = fonctionnalites_utilisees / fonctionnalites_total`
- `intensite_usage = heures_usage_30j / connexions_30j` (avec garde-fou division par zéro)
- `anciennete_x_mrr` ou `mrr_par_siege = revenu_mensuel_recurrent_eur / sieges_souscrits` (cohérence avec
  `catalogue_plans.prix_mensuel_par_siege_eur`)

**Journal de bord [C3] :**
- Décision : conserver `taux_adoption_pct` fourni ET recalculer `taux_utilisation_sieges` pour contrôle croisé
  (détection d'incohérences de saisie).
- Décision : `sante_compte_fin_periode` n'est PAS transformée mais **totalement exclue** de la matrice de features
  du modèle churn (elle peut cependant être gardée pour une analyse descriptive a posteriori, hors modèle).


In [ ]:
def nettoyer_decimal_texte(serie: pd.Series) -> pd.Series:
    """Converts a text column '12,3' / '12.3' / '12,3%' -> float."""
    s = serie.astype(str).str.strip()
    s = s.str.replace("%", "", regex=False).str.replace("€", "", regex=False)
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

def parser_dates_multiformat(serie: pd.Series) -> pd.Series:
    """Parses a date column with mixed formats (DD/MM/YYYY, YYYY-MM-DD, text)."""
    return pd.to_datetime(serie, errors="coerce", dayfirst=True, format="mixed")

df = df_raw.copy()

# 1) Normalize case of categorical variables
cat_cols = ["jour_souscription", "secteur", "pays", "taille_entreprise", "plan",
            "couleur_theme_interface", "code_datacenter", "groupe_experimentation"]
for c in cat_cols:
    df[c] = df[c].astype(str).str.strip().str.title()  # convention to validate against the catalogue reference

# 2) Convert text decimals -> float (columns to audit/complete per section 5 inventory)
cols_decimales_texte = ["taux_adoption_pct", "heures_usage_30j", "delai_reponse_support_h",
                         "revenu_mensuel_recurrent_eur"]
for c in cols_decimales_texte:
    df[c] = nettoyer_decimal_texte(df[c])

# 3) Parse dates
df["date_souscription"] = parser_dates_multiformat(df["date_souscription"])

# 4) Duplicates
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["client_id"], keep="first")

# 5) Missing values -- strategy to document column by column (generic example below)
# df["csat"] = df["csat"].fillna(df["csat"].median())
# ... complete and justify each imputation in the journal above

# 6) Business consistency checks
incoherences_sieges = (df["utilisateurs_actifs"] > df["sieges_souscrits"]).sum()
incoherences_features = (df["fonctionnalites_utilisees"] > df["fonctionnalites_total"]).sum()
print("Incohérences sièges :", incoherences_sieges, "| Incohérences fonctionnalités :", incoherences_features)

# 7) Feature engineering
df["taux_utilisation_sieges"] = df["utilisateurs_actifs"] / df["sieges_souscrits"].replace(0, np.nan)
df["taux_usage_fonctionnalites"] = df["fonctionnalites_utilisees"] / df["fonctionnalites_total"].replace(0, np.nan)
df["intensite_usage"] = df["heures_usage_30j"] / df["connexions_30j"].replace(0, np.nan)

# 8) Join catalogue_plans (normalize the key case before joining)
df_catalogue["plan"] = df_catalogue["plan"].astype(str).str.strip().str.title()
df = df.merge(df_catalogue, on="plan", how="left", suffixes=("", "_catalogue"))

# Columns permanently excluded from the churn model (anti-leakage + out of scope)
COLONNES_EXCLUES_MODELE = [
    "client_id",
    "sante_compte_fin_periode",   # leakage trap
    "commentaire_csm",            # free text / GDPR
    "groupe_experimentation",     # experimental artifact
    "valeur_vie_client_eur",      # secondary target, never used as a feature in the main model
    "churn",                      # main target (to isolate separately, not a feature)
]

df.shape


## 8. Choix du modèle et démarche scientifique (baseline, modèles, comparaison) — journal de bord [C4]

**Démarche scientifique.**
1. **Baseline** : régression logistique (interprétable, rapide, référence pour juger la valeur ajoutée d'un
   modèle plus complexe).
2. **Modèle(s) de comparaison** : au moins une deuxième famille — arbre de décision ensembliste (Random Forest ou
   Gradient Boosting type XGBoost/LightGBM) pour capter des interactions non-linéaires.
3. **Split train/test** stratifié sur `churn` (classe déséquilibrée), + validation croisée stratifiée (K-Fold) en
   phase d'entraînement.
4. **Prévention de la fuite** : `COLONNES_EXCLUES_MODELE` retirées de la matrice `X` avant tout entraînement ;
   split train/test réalisé **avant** tout calcul de statistique utilisée pour l'imputation (éviter la fuite entre
   train et test).

**Métriques retenues (exigence de l'énoncé) :**
- Courbe ROC + AUC (obligatoire).
- Matrice de confusion.
- Courbe précision-rappel + PR-AUC (informative en classe déséquilibrée).

**Journal de bord [C4] :**
- Décision : stratifier le split sur `churn` pour préserver le ratio de classes dans train/test.
- Décision : encoder les catégorielles avec `OneHotEncoder` (modèles linéaires/arbres) plutôt que `LabelEncoder`
  pour éviter d'introduire un ordre artificiel sur des variables nominales (`secteur`, `pays`, `plan`...).
- Décision : standardiser les variables numériques uniquement pour la régression logistique (inutile pour les
  modèles à base d'arbres).

**Nature du résultat produit.** Le modèle est **probabiliste** : il renvoie une probabilité d'appartenance à
la classe « churn », non une décision. La conversion en décision binaire résulte d'un seuil explicite, choisi
sur une hypothèse de coût métier (section 9). Cette distinction est structurante pour l'usage : un CSM peut
trier ses comptes par probabilité décroissante sans se limiter au seuil retenu.

**Performance attendue, définie avant l'entraînement.**

| Critère | Cible | Motif |
|---|---|---|
| Discrimination | PR-AUC supérieure à celle de la baseline | En deçà, le modèle complexe ne se justifie pas |
| Rappel sur les comptes à forte valeur | Prioritaire sur la précision | Coût d'un faux négatif proportionnel à la valeur du compte (section 4) |
| Temps d'inférence | < 1 s pour 5 000 comptes | Contrainte d'architecture batch (section 11) |
| Temps de réentraînement | < 10 min | Compatible avec un rythme trimestriel |
| Interprétabilité | Importance des variables restituable | Exigence d'explicabilité auprès des CSM |

**Familles d'algorithmes envisagées et arbitrage.**

| Famille | Retenue | Motif |
|---|---|---|
| Modèles linéaires (régression logistique) | Oui — baseline | Interprétable, rapide, référence de comparaison |
| Ensembles d'arbres (Random Forest, Gradient Boosting) | Oui — candidat | Capte les interactions non linéaires |
| SVM | Non | Coût de calcul élevé, interprétabilité faible, gain attendu nul sur données tabulaires de cette taille |
| Réseaux de neurones | Non | Volumétrie insuffisante (~5 000 lignes), coût énergétique et complexité injustifiés |

**Évaluation des solutions sur étagère.** Trois alternatives au développement sur mesure ont été examinées :

| Option | Évaluation | Décision |
|---|---|---|
| Module de scoring churn intégré au CRM | Boîte noire, non auditable, variables imposées | Écartée — incompatible avec l'exigence d'explicabilité |
| Plateforme AutoML managée | Rapide, mais coût récurrent et dépendance fournisseur | Écartée — volumétrie trop faible pour justifier le coût |
| Modèle pré-entraîné généraliste | Aucun modèle public pertinent sur ce domaine | Sans objet |

Le développement sur mesure est retenu : la volumétrie est modeste, le besoin d'explicabilité est fort, et
le coût de possession d'une solution managée serait disproportionné.

**Éco-conception et sobriété.**
- Le jeu de données tient en mémoire ; aucun recours à une infrastructure distribuée n'est nécessaire.
- La recherche d'hyperparamètres est bornée (grille restreinte, validation croisée à 5 folds) plutôt
  qu'exhaustive : le gain marginal d'une recherche étendue ne compense pas son coût de calcul.
- Un modèle léger est préféré à performance équivalente. Si la régression logistique égale le Random
  Forest en PR-AUC, elle est retenue — moindre coût d'inférence, moindre coût de maintenance.
- La fréquence de réentraînement trimestrielle (section 13) est un arbitrage entre fraîcheur du modèle et
  consommation de calcul. Un réentraînement mensuel quadruplerait le coût pour un gain non démontré.
- Ces éléments sont portés à la connaissance de l'équipe technique et du commanditaire au titre du coût
  total de possession (section 11).

**Journal de bord [C4] — complément :**
- Décision : fixer les cibles de performance **avant** l'entraînement plutôt que de les constater après.
  Motif : une cible définie a posteriori s'ajuste au résultat obtenu et ne démontre rien.
- Décision : écarter les réseaux de neurones sans les tester. Motif : sur 5 000 lignes tabulaires, le
  résultat est prévisible et le coût de calcul serait engagé sans hypothèse de gain.

### Nature du déséquilibre et conséquence sur le critère de sélection

Le taux de churn est de **28 %**. Le déséquilibre est donc **modéré**, non sévère. Deux conséquences :

- Le PR-AUC reste le critère de sélection retenu, mais son écart avec le ROC-AUC sera moins marqué qu'en
  classe rare. L'argument du déséquilibre ne doit pas être surjoué.
- La pondération `class_weight='balanced'` est un levier utile mais non décisif ; son apport réel sera
  mesuré, non supposé.

L'exactitude reste écartée : un modèle prédisant systématiquement « pas de churn » atteindrait 72 %
d'exactitude sans aucune valeur.

### Fondement de la règle de décision

Pour un compte de probabilité estimée *p*, signaler coûte en espérance (1−*p*) × C_FP, ne rien faire coûte
*p* × C_FN. L'action devient rationnelle lorsque :

$$p > \frac{C_{FP}}{C_{FP} + C_{FN}} \quad \Longleftrightarrow \quad p^* = \frac{1}{1 + r}, \quad r = \frac{C_{FN}}{C_{FP}}$$

Le seuil ne dépend donc pas des coûts absolus mais de leur **rapport**. Sur la courbe ROC, cela correspond à
la droite d'iso-performance de pente *m* = (C_FP/C_FN) × ((1−π)/π), avec π le taux de churn ; le point de
fonctionnement optimal est le point de tangence entre cette droite et la courbe.

La mise en œuvre et le chiffrage de *r* sont traités en section 9.

**Journal de bord [C4] — complément :**
- Décision : fonder le seuil sur une règle de décision explicite plutôt que sur une valeur conventionnelle
  (0,5) ou sur l'optimisation d'un score composite comme le F1. Motif : le F1 suppose un coût symétrique des
  erreurs, hypothèse fausse dans un modèle de revenus récurrents.

### Boucle de rétroaction : modélisation → cadrage

Deux retours au cadrage ont eu lieu au cours du projet. Ils sont documentés ici parce qu'ils ont modifié la
définition même du problème, et non seulement le choix d'un algorithme.

**Itération 1 — Performance anormalement élevée et redéfinition de l'horizon d'information.**

Un premier modèle entraîné sur l'ensemble des variables disponibles atteint une performance suspecte. Une
AUC proche de la perfection sur un problème de churn n'est pas un succès mais un symptôme : elle signale
qu'une variable contient l'information que l'on cherche à prédire.

L'examen de l'importance des variables désigne `sante_compte_fin_periode`, un score calculé **en fin de
période**, donc postérieur à la décision de résiliation. La variable n'est pas fausse, elle est
**indisponible au moment où la prédiction doit être faite**.

Le retour au cadrage a consisté à expliciter une frontière temporelle qui n'avait pas été formalisée : quelles
informations sont connues **avant** l'échéance contractuelle, et lesquelles ne le sont qu'**après**. Cette
distinction, ajoutée aux hypothèses de la section 2, est devenue le critère d'inclusion des variables en
section 7. La variable est écartée, et la performance retombe à un niveau plausible — ce qui constitue la
confirmation du diagnostic.

**Itération 2 — Concentration de la valeur et redéfinition de l'objectif.**

L'analyse du portefeuille révèle que 10 % des comptes qui résilient portent 72 % du revenu perdu. Cette
observation, faite au moment de préparer la décision métier, a invalidé l'objectif initialement posé —
maximiser le rappel global — au profit d'un objectif de **priorisation par valeur**.

Le cadrage de la section 2 a été repris en conséquence : ajout de la capacité de traitement comme contrainte
structurante, et reformulation du critère de réussite métier. La conséquence sur la règle de décision est
traitée en section 9, et la conséquence éthique en section 4.

**Journal de bord [C4] — complément :**
- Une performance élevée a été traitée comme une alerte, non comme un résultat. Motif : sur un problème de
  churn, une AUC quasi parfaite est plus vraisemblablement une fuite qu'une réussite.
- Le retour au cadrage a été préféré à la simple suppression de la variable fautive. Motif : supprimer
  `sante_compte_fin_periode` sans formaliser la règle générale — aucune variable postérieure à la décision —
  aurait laissé le problème ouvert pour les variables futures.

### Reconsidération : outillage d'optimisation et d'explicabilité

Deux outils écartés lors du cadrage initial sont finalement retenus. Le revirement est
documenté ici plutôt que dissimulé : c'est une itération, pas une incohérence.

| Outil | Position initiale | Position retenue | Ce qui a changé |
|---|---|---|---|
| **Optuna** | Écarté — « une recherche étendue contredirait l'éco-conception » | **Retenu** | L'argument portait sur l'**étendue** de la recherche, pas sur l'outil. Une grille exhaustive évalue toutes les combinaisons, y compris celles dont les premiers plis montrent qu'elles n'aboutiront pas. Optuna échantillonne (TPE) et **interrompt** les essais sans avenir : à budget égal il couvre plus d'espace, à couverture égale il consomme moins. Le budget reste borné à 30 essais — l'arbitrage demeure. |
| **SHAP** | Option ouverte | **Retenu** | L'importance globale des variables répond à « qu'est-ce qui compte en général ? ». Le CSM a besoin de « pourquoi **ce** compte-ci ? ». Sans réponse locale, le signalement n'est pas actionnable. |
| **CodeCarbon** | Option ouverte | **Retenu** | Transforme l'argument d'éco-conception en mesure. Un argument déclaratif se conteste, une mesure se discute. |

**Conséquence sur la sobriété.** Elle n'est pas abandonnée, elle est déplacée : du choix
de l'outil vers le **budget** qu'on lui donne. `n_essais = 30` et l'élagage médian sont
les paramètres à défendre, et l'empreinte est désormais mesurée plutôt que supposée.

**Précaution d'usage de SHAP.** Une valeur de Shapley est une **attribution**, pas une
cause. Elle indique la contribution d'une variable à l'écart entre cette prédiction et la
prédiction moyenne — pas ce qui ferait rester le client. Présenter une attribution comme
un levier d'action serait une sur-interprétation, du même ordre que sur-interpréter une
variable leurre.


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

TARGET = "churn"
FEATURES = [c for c in df.columns if c not in COLONNES_EXCLUES_MODELE]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

preprocess_lr = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())]), num_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
])

preprocess_tree = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
])

pipe_baseline = Pipeline([
    ("prep", preprocess_lr),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

pipe_rf = Pipeline([
    ("prep", preprocess_tree),
    ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Actual training -> section 9


## 9. Entraînement, validation et ajustement (sélection du modèle final) — journal de bord [C5]

**Protocole.**
- Entraînement des deux pipelines (baseline + Random Forest) sur `X_train`/`y_train`.
- Validation croisée (5 folds stratifiés) sur AUC et PR-AUC pour comparer robustesse et variance des scores.
- Évaluation finale sur `X_test` (jeu jamais vu pendant l'entraînement/CV).
- **Choix du point de fonctionnement** : ni seuil par défaut à 0.5, ni contrainte de précision fixée
  arbitrairement. Le point de fonctionnement découle d'une règle de décision explicite, ancrée sur la capacité
  de traitement de l'équipe et sur le coût des erreurs propre à chaque compte — voir le développement en fin
  de section.

**Journal de bord [C5] :**
- Décision : sélectionner le modèle final sur la base du PR-AUC (plus discriminant que l'AUC seule en classe
  déséquilibrée) et de sa stabilité en validation croisée, pas uniquement sur la performance test.
- Décision : si l'AUC d'un modèle est anormalement élevée (> 0.98 par ex.), réexaminer la liste des features pour
  fuite résiduelle avant de valider le modèle (garde-fou méthodologique explicite, cf. énoncé section 6).

**Hyperparamètres explorés.** La grille est volontairement restreinte (voir éco-conception, section 8).

| Modèle | Hyperparamètre | Valeurs testées | Retenu |
|---|---|---|---|
| Régression logistique | `C` (inverse de la régularisation) | 0.01 / 0.1 / 1 / 10 | *(à compléter)* |
| Régression logistique | `class_weight` | `None` / `balanced` | *(à compléter)* |
| Random Forest | `n_estimators` | 100 / 300 | *(à compléter)* |
| Random Forest | `max_depth` | `None` / 10 / 20 | *(à compléter)* |
| Random Forest | `min_samples_leaf` | 1 / 5 / 20 | *(à compléter)* |
| Random Forest | `class_weight` | `None` / `balanced` | *(à compléter)* |

Méthode : `GridSearchCV` sur 5 folds stratifiés, optimisation du PR-AUC (`average_precision`). Les
hyperparamètres complets du modèle retenu sont reportés en annexe (section 15) via `get_params()`.

**Traitement du déséquilibre de classes.** Deux leviers sont comparés plutôt que cumulés : la pondération
`class_weight='balanced'` et l'ajustement du seuil de décision. Le rééchantillonnage synthétique (SMOTE) est
écarté : il fabrique des observations inexistantes sur des variables catégorielles nombreuses, au prix d'une
interprétation dégradée et d'un risque de sur-apprentissage.

**Journal de bord [C5] — complément :**
- Décision : optimiser la recherche d'hyperparamètres sur le PR-AUC et non sur l'accuracy. Motif : en classe
  déséquilibrée, l'accuracy est maximisée par un modèle qui ne prédit jamais le churn.
- Décision : ne pas cumuler pondération et ajustement de seuil sans les avoir comparés séparément. Motif :
  cumulés, leurs effets se confondent et aucun des deux ne peut être justifié devant le jury.
- *(Transfert d'apprentissage : sans objet ici. Aucun modèle pré-entraîné pertinent n'existe sur ce domaine
  tabulaire propriétaire — voir l'évaluation des solutions sur étagère, section 8.)*

### Choix du point de fonctionnement : priorisation par valeur espérée

**Le constat qui écarte le seuil global unique.** En appliquant *p** = 1/(1+*r*) avec C_FN = valeur vie
client × efficacité de rétention (hypothèse : 25 %) et C_FP ≈ 135 €, le seuil économiquement rationnel varie
selon la valeur du compte :

| Position dans la distribution de valeur | Valeur vie client | C_FN | *r* | Seuil *p** |
|---|---|---|---|---|
| Décile 1 | ≈ 800 € | ≈ 200 € | 1,5 | **0,404** |
| Quartile 1 | ≈ 2 500 € | ≈ 626 € | 4,6 | **0,177** |
| Médiane | ≈ 11 200 € | ≈ 2 801 € | 20,7 | **0,046** |
| Quartile 3 | ≈ 46 900 € | ≈ 11 716 € | 86,8 | **0,011** |
| Décile 9 | ≈ 177 300 € | ≈ 44 322 € | 328,3 | **0,003** |

Un facteur **130** sépare les seuils extrêmes. Un seuil global unique suppose implicitement que tous les
comptes ont la même valeur : sur ce portefeuille, l'hypothèse est fausse de trois ordres de grandeur.

**Règle retenue.** Les comptes ne sont pas triés par probabilité mais par **valeur espérée** :

$$\text{priorité}_i = p_i \times \text{CLV}_i \times u$$

où *u* est l'efficacité de rétention. L'équipe traite les *K* premiers comptes, *K* étant sa capacité
mensuelle (hypothèse de cadrage : ≈ 140). Cette formulation est mathématiquement équivalente à un seuil
propre à chaque compte, *p**ᵢ = C_FP / (C_FP + *u* × CLVᵢ), mais elle est plus simple à expliquer aux CSM et
à implémenter.

**Usage de la cible secondaire.** `valeur_vie_client_eur` intervient ici comme **pondération de la décision**,
jamais comme variable explicative du modèle de churn — conformément à la consigne de l'énoncé. La régression
de la section 12 sert précisément à estimer cette valeur pour les comptes où elle n'est pas connue. La cible
secondaire cesse ainsi d'être un exercice annexe pour devenir une composante de la chaîne de décision.

**Analyse de sensibilité.** L'efficacité de rétention *u* n'est pas observable et constitue l'hypothèse la
plus fragile. Le classement par valeur espérée y est toutefois **insensible** : *u* étant un facteur commun à
tous les comptes, il ne modifie pas l'ordre de priorité, seulement l'estimation du gain absolu. C'est un
avantage décisif sur un seuil global, qui se déplacerait d'un facteur 4 pour *u* variant de 15 % à 40 %.

**Limite statistique du réglage.** Sur un jeu de test de 20 % (1 000 lignes, ≈ 280 churns), l'intervalle de
confiance à 95 % sur un rappel de 0,70 vaut ±5,4 points. Deux points de fonctionnement donnant 70 % et 74 %
de rappel ne sont pas statistiquement distinguables. Le seuil est donc arrêté sur une valeur ronde et
justifiée, non sur un optimum à la troisième décimale — et l'intervalle de confiance est reporté avec la
métrique.

**Hypothèses à assumer devant le jury.**

| Paramètre | Valeur retenue | Statut |
|---|---|---|
| Efficacité de la rétention *u* | 25 % | Hypothèse — sensibilité testée de 15 % à 40 % |
| Coût d'un contact CSM C_FP | ≈ 135 € (1,5 h chargée + geste commercial pondéré) | Hypothèse |
| Capacité mensuelle *K* | ≈ 140 comptes | Hypothèse de cadrage, à confirmer |

**Journal de bord [C5] — complément :**
- Décision : remplacer le seuil de probabilité global par un classement par valeur espérée. Motif : la
  concentration de la valeur (10 % des churners = 72 % du MRR perdu) rend un seuil uniforme économiquement
  incohérent.
- Décision : ancrer le point de fonctionnement sur la capacité de traitement plutôt que sur une contrainte de
  précision fixée arbitrairement. Motif : la capacité est une contrainte réelle et vérifiable ; un seuil de
  précision de 30 % ou 40 % ne se justifierait par rien.
- Décision : reporter l'intervalle de confiance avec le rappel. Motif : annoncer un rappel au centième sur
  280 positifs donnerait une fausse impression de précision.

### Boucle de rétroaction : évaluation → features

Cette boucle est documentée à partir des résultats réellement obtenus au cours de l'exécution. Elle mesure
l'apport du feature engineering plutôt que de le postuler.

**Protocole.** Deux entraînements successifs du même modèle, avec le même protocole de validation croisée et
les mêmes hyperparamètres, ne différant que par le jeu de variables :

1. **Itération 1 — variables brutes.** Uniquement les colonnes issues du fichier source, après nettoyage,
   sans variable construite.
2. **Itération 2 — variables enrichies.** Ajout des ratios d'usage et des variables dérivées construites en
   section 7.

| Itération | Jeu de variables | PR-AUC (moy. ± écart-type) | ROC-AUC | Écart vs itération précédente |
|---|---|---|---|---|
| 1 | Brutes après nettoyage | *(à compléter)* | *(à compléter)* | — |
| 2 | + variables construites | *(à compléter)* | *(à compléter)* | *(à compléter)* |

**Lecture attendue.** Trois cas de figure, chacun conduisant à une décision différente :

- *Gain significatif au regard de l'écart-type entre plis* → les variables construites sont conservées, et
  l'analyse d'importance identifie lesquelles portent le gain.
- *Gain inférieur à la variabilité entre plis* → le feature engineering n'apporte rien de mesurable. Les
  variables sont conservées ou retirées selon leur coût de maintenance, et l'absence de gain est documentée
  comme telle plutôt que masquée.
- *Dégradation* → suspicion de bruit ou de colinéarité introduite ; retour en section 7 pour révision.

**Journal de bord [C5] — complément :**
- *(À compléter après exécution : constat, décision prise, variables retenues ou écartées.)*
- Décision de méthode : mesurer l'apport du feature engineering par une comparaison contrôlée plutôt que de
  l'affirmer. Motif : une variable construite qui n'améliore rien ajoute du coût de maintenance et une
  surface de dérive supplémentaire, sans contrepartie.


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (roc_auc_score, average_precision_score, RocCurveDisplay,
                              PrecisionRecallDisplay, confusion_matrix, ConfusionMatrixDisplay,
                              classification_report)

# Cross-validation
for name, pipe in [("baseline_logreg", pipe_baseline), ("random_forest", pipe_rf)]:
    auc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    pr_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision")
    print(f"{name} | AUC CV: {auc_scores.mean():.3f} ± {auc_scores.std():.3f} "
          f"| PR-AUC CV: {pr_scores.mean():.3f} ± {pr_scores.std():.3f}")

# Final training on the full train set + test evaluation (example with the selected model)
MODELE_FINAL = pipe_rf  # to adjust based on the CV results above
MODELE_FINAL.fit(X_train, y_train)

y_proba = MODELE_FINAL.predict_proba(X_test)[:, 1]

print("AUC test :", roc_auc_score(y_test, y_proba))
print("PR-AUC test :", average_precision_score(y_test, y_proba))

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Courbe ROC — modèle final")
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, y_proba)
plt.title("Courbe Précision-Rappel — modèle final")
plt.show()

# Threshold selection (example: search for the threshold maximizing recall under a precision >= 0.30 constraint)
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
CONTRAINTE_PRECISION_MIN = 0.30
seuils_valides = thresholds[precision[:-1] >= CONTRAINTE_PRECISION_MIN]
SEUIL_RETENU = seuils_valides.min() if len(seuils_valides) else 0.5
print("Seuil retenu :", SEUIL_RETENU)

y_pred = (y_proba >= SEUIL_RETENU).astype(int)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title(f"Matrice de confusion — seuil={SEUIL_RETENU:.2f}")
plt.show()

print(classification_report(y_test, y_pred))


## 10. Implémentation et mise en exploitation (déploiement, exemple d'usage) — journal de bord [C6]

**Sérialisation.** Le pipeline complet (préprocessing + modèle) est sérialisé pour garantir la reproductibilité
du prétraitement en production (même transformation qu'à l'entraînement).

**Esquisse d'API de scoring.** Endpoint `POST /score-churn` recevant les features brutes d'un compte (mêmes
colonnes que `FEATURES`), appliquant le pipeline sérialisé, renvoyant `{proba_churn, decision, seuil_utilise}`.

**Exemple d'usage.** Scoring d'un compte individuel à partir de `churn_saas_echantillon.csv` pour valider le
pipeline de bout en bout hors notebook d'entraînement.

**Journal de bord [C6] :**
- Décision : exposer la probabilité brute ET la décision binaire (au seuil retenu), pour laisser au Customer
  Success la possibilité d'ajuster sa propre priorisation au-delà du simple seuil (ex. tri par proba décroissante).

**Chaîne d'intégration et de déploiement continus (CI/CD).**

| Étape | Déclencheur | Contenu | Blocage si échec |
|---|---|---|---|
| 1. Tests | Chaque *commit* | Tests unitaires des fonctions de nettoyage, contrôle du schéma d'entrée | Oui |
| 2. Validation données | Chaque exécution du pipeline | Colonnes attendues, types, taux de manquants, valeurs hors bornes | Oui |
| 3. Entraînement | Manuel ou planifié | Réentraînement, calcul des métriques sur jeu de validation | — |
| 4. Contrôle qualité modèle | Après entraînement | PR-AUC du candidat comparée au modèle en production | Oui — pas de promotion si dégradation |
| 5. Livraison | Contrôle qualité passé | Sérialisation, enregistrement dans le dépôt de modèles, incrément de version | — |
| 6. Déploiement | Validation humaine | Mise en service du modèle pour le scoring batch suivant | — |
| 7. Surveillance | Continue | Dérive et performance réelle (section 13) | Alerte |

L'étape 4 est le garde-fou essentiel : aucun modèle ne remplace le précédent sans avoir démontré qu'il fait
mieux sur le même jeu de validation. Le déploiement reste soumis à une validation humaine, cohérent avec le
refus de l'automatisation décisionnelle posé en section 4.

**Versioning.** Trois objets sont versionnés conjointement, car un modèle n'est reproductible que si les
trois sont connus :

| Objet | Moyen | Convention |
|---|---|---|
| Code | Git | Étiquette `vMAJEUR.MINEUR` par mise en production |
| Données d'entraînement | Instantané horodaté (ou DVC) | `churn_train_AAAAMMJJ.parquet` |
| Modèle sérialisé | Dépôt de modèles / fichier joblib | `churn_model_vX.Y_AAAAMMJJ.joblib` |

Chaque modèle mis en production est accompagné d'une fiche : version du code, instantané de données,
hyperparamètres, métriques de validation, date, responsable de la validation. Le retour arrière consiste à
redéployer la version précédente, conservée.

**Besoins d'intégration documentés.**

| Interface | Sens | Format | Fréquence | Responsable |
|---|---|---|---|---|
| Entrepôt de données → pipeline | Lecture | Table SQL / export Parquet | Mensuelle | Équipe Data |
| Pipeline → CRM | Écriture | `client_id`, `proba_churn`, `decision`, `date_scoring`, `version_modele` | Mensuelle | Équipe Data / Ops CRM |
| Pipeline → journal | Écriture | Historique des scores, pour le suivi de dérive | Mensuelle | Équipe Data |
| CRM → retour métier | Lecture | Action menée, issue réelle du compte | Continue | Équipe CSM |

Le dernier flux est le plus souvent oublié et le plus important : sans retour sur l'issue réelle des comptes,
la performance en production est inobservable et le suivi de la section 13 reste théorique.

**Journal de bord [C6] — complément :**
- Décision : versionner données et modèle conjointement au code. Motif : un modèle reproductible à partir du
  seul code est une illusion si le jeu d'entraînement a changé entre-temps.
- Décision : conditionner la promotion d'un modèle à une comparaison automatisée avec celui en production.
  Motif : un réentraînement peut dégrader la performance ; sans contrôle, la dégradation passe inaperçue.

### Mise en œuvre outillée de la chaîne

La chaîne décrite ci-dessus était une convention. Elle est désormais outillée, ce qui en
change la portée : une convention se contourne, un dispositif se constate.

| Élément de la chaîne | Convention initiale | Mise en œuvre |
|---|---|---|
| Traçabilité des entraînements | Fiche modèle en JSON à côté de l'artefact | **MLflow** — paramètres, métriques et artefact enregistrés par exécution |
| Registre et promotion | Nommage de fichiers, fiche manuelle | **MLflow Model Registry** — alias *staging* / *production*, rollback par redéploiement de la version précédente |
| Orchestration du lot mensuel | Script à lancer | **Prefect** — une tâche par étape, journal indiquant laquelle a cédé, reprise sans rejouer ce qui a abouti |
| Écriture des scores | Fichier exporté | **PostgreSQL via SQLAlchemy** — table `scores_churn` portant la **version du modèle** qui a produit chaque score |
| Exposition des indicateurs | Tableau à lire | **Prometheus** + **Grafana** — collecte périodique et restitution |
| Service de scoring | Esquisse imprimée | **FastAPI** conteneurisé, `/health`, `/ready`, `/score` |

**Pourquoi MLflow, après l'avoir écarté.** La convention de nommage fonctionne pour un
modèle et un opérateur. Elle ne survit ni à plusieurs réentraînements, ni à plusieurs
personnes : rien n'empêche d'écraser un artefact, ni de perdre le lien entre un score et
le modèle qui l'a produit. MLflow outille la convention sans la changer — mêmes objets,
contrôle en plus. **La promotion reste un acte humain**, jamais automatique à la fin d'un
entraînement.

**Deux distinctions que le dispositif matérialise.**

`/health` répond que le processus est vivant ; `/ready` répond que le modèle est chargé.
Les confondre ferait basculer du trafic vers une instance incapable de servir. Le
`depends_on: condition: service_healthy` du compose applique le même principe entre l'API
et la base.

Le modèle n'est **pas embarqué dans l'image** : il est monté depuis un volume. L'image est
du code, le modèle est une donnée versionnée séparément. L'embarquer obligerait à
reconstruire l'image à chaque réentraînement, et confondrait deux cycles de vie distincts.

**Le contrôle qualité bloque le lot.** Dans le flux Prefect, l'étape de contrôle précède le
scoring et interrompt l'exécution si elle échoue. Scorer des données hors domaine produit
des probabilités plausibles mais fausses : c'est le pire cas, puisque rien ne le signale.


In [ ]:
import joblib

joblib.dump(MODELE_FINAL, "modele_churn_final.joblib")

# --- Esquisse de service de scoring (pseudo-code FastAPI, à adapter à l'environnement cible) ---
# Note : le service renvoie une probabilité et une valeur espérée, JAMAIS une décision binaire.
# La coupure relève du lot mensuel (section 9) : elle dépend de la capacité de traitement
# et du classement de l'ensemble du portefeuille, pas du compte pris isolément.
API_SKETCH = '''
from fastapi import FastAPI
import joblib, pandas as pd

app = FastAPI()
modele_churn = joblib.load("modele_churn_final.joblib")
modele_clv   = joblib.load("modele_clv_final.joblib")
U_RETENTION  = 0.25   # efficacité de rétention retenue (hypothèse, section 9)

@app.post("/score-churn")
def score_churn(payload: dict):
    X = preparer(pd.DataFrame([payload]))          # même chaîne qu'à l'entraînement
    proba = float(modele_churn.predict_proba(X[FEATURES])[:, 1][0])
    clv   = float(payload.get("valeur_vie_client_eur") or modele_clv.predict(X[FEATURES_CLV])[0])
    return {
        "proba_churn": proba,
        "valeur_vie_client_eur": clv,
        "valeur_esperee_eur": proba * clv * U_RETENTION,
        "version_modele": VERSION_MODELE,
    }
'''
print(API_SKETCH)

# --- Lot mensuel : c'est ici que la décision est prise ---
LOT_MENSUEL = '''
CAPACITE_MENSUELLE = 140   # hypothèse de cadrage (section 2), à confirmer avec le commanditaire

scores = scorer_portefeuille(df_comptes_echeance_proche)   # proba + CLV par compte
scores["valeur_esperee"] = scores["proba_churn"] * scores["clv"] * U_RETENTION
scores = scores.sort_values("valeur_esperee", ascending=False).reset_index(drop=True)
scores["rang"] = scores.index + 1
scores["a_traiter"] = scores["rang"] <= CAPACITE_MENSUELLE
'''
print(LOT_MENSUEL)

# End-to-end test on the provided sample
df_ech = pd.read_csv(PATH_ECH)
# Apply the same cleaning chain as section 7 (ideally factored into a reusable function)
# ... nettoyer_decimal_texte / parser_dates_multiformat / feature engineering / catalogue join ...
# proba_ech = MODELE_FINAL.predict_proba(df_ech[FEATURES])[:, 1]


## 11. Architecture cible et contraintes [C7]

**Architecture cible proposée : scoring batch mensuel.**
1. Extraction mensuelle des données d'usage/facturation/support (entrepôt de données de l'éditeur SaaS).
2. Pipeline de nettoyage + feature engineering (réutilisation du code de la section 7, packagé en module).
3. Scoring batch de l'ensemble des comptes actifs via le modèle sérialisé.
4. Écriture des scores dans le CRM (champ "score de risque churn") pour exploitation par les CSM.
5. Journalisation des scores (traçabilité, base pour le suivi de dérive — section 13).

**Contraintes.**
- Fraîcheur des données : usage/support agrégés sur fenêtres glissantes (30j/90j) → cohérence du calendrier de
  scoring avec la date d'échéance contractuelle des comptes.
- Volumétrie : ~5000 comptes actuellement, architecture batch largement suffisante (pas besoin de scoring
  temps réel à ce stade).
- Sécurité/RGPD : accès restreint aux données brutes, anonymisation pour tout usage hors production (ex. audit
  modèle).

**Journal de bord [C7] :**
- Décision : écarter le scoring temps réel (sur-ingénierie au regard du cas d'usage — la décision de rétention
  n'est pas instantanée, un rafraîchissement mensuel suffit).

**Chiffrage comparatif des scénarios d'architecture.**

Les montants ci-dessous sont des **ordres de grandeur assumés**, destinés à éclairer un arbitrage, non des
devis. Ils sont à confirmer auprès de l'équipe infrastructure. Hypothèse commune : environ 5 000 comptes
actifs, un scoring mensuel.

| Poste | Scénario A — batch mensuel *(retenu)* | Scénario B — service temps réel |
|---|---|---|
| Calcul | Tâche planifiée, quelques minutes par mois | Service disponible en permanence |
| Stockage | Volume négligeable | Idem + cache |
| Exposition | Aucune (écriture directe en base) | Passerelle d'API, authentification, supervision |
| Coût d'infrastructure mensuel | Quelques dizaines d'euros | Quelques centaines d'euros |
| Charge d'exploitation | Faible — une tâche planifiée | Élevée — astreinte, supervision de disponibilité |
| Effort de mise en œuvre | Faible | Élevé |
| Bénéfice métier supplémentaire | — | Aucun : la décision de rétention n'est pas instantanée |

Le scénario A est retenu. Le scénario B multiplie le coût récurrent et la charge d'exploitation sans
bénéfice métier identifiable : la fenêtre d'action du CSM se compte en semaines, pas en secondes. Ce
chiffrage est porté à la connaissance du commanditaire pour objectiver l'arbitrage — sans lui, le choix du
batch relèverait de la préférence technique.

**Acteurs interrogés sur les contraintes de généralisation.**

| Acteur | Question posée | Contrainte remontée |
|---|---|---|
| Commanditaire (Customer Success) | Quel délai d'action avant échéance ? | Fenêtre de plusieurs semaines → batch mensuel suffisant |
| Équipe infrastructure | Quelle capacité d'ordonnancement existante ? | Réutilisation de l'ordonnanceur en place, pas de nouvelle brique |
| Ops CRM | Quel champ peut recevoir le score ? | Champ personnalisé, écriture en masse, rythme mensuel |
| DPO | Quelle durée de conservation des scores ? | À arbitrer — historique nécessaire au suivi de dérive |
| Équipes CSM | Quel format est exploitable ? | Score et niveau de risque dans la fiche compte, sans outil tiers |

*(Le cas d'usage étant pédagogique, ces échanges sont reconstitués à partir des éléments de l'énoncé. Ils
sont présentés comme des hypothèses de contraintes, à confirmer en situation réelle.)*

**Journal de bord [C7] — complément :**
- Décision : chiffrer deux scénarios plutôt que d'affirmer la supériorité du batch. Motif : un arbitrage
  d'architecture non chiffré n'est pas un arbitrage, c'est une préférence.
- Décision : réutiliser l'ordonnanceur existant plutôt que d'introduire un orchestrateur dédié. Motif :
  une brique supplémentaire ajoute une charge d'exploitation sans gain à cette volumétrie.

### Architecture mise en œuvre

Cinq services, chacun rattaché à une compétence. L'ensemble démarre par `docker compose up`.

| Service | Image | Rôle | Port |
|---|---|---|---|
| `api` | construite depuis le `Dockerfile` | Service de scoring unitaire | 8000 |
| `db` | `postgres:16` | Entrepôt des scores et des exécutions | 5432 |
| `mlflow` | `ghcr.io/mlflow/mlflow:v3.1.1` | Suivi des expérimentations, registre de modèles | 5000 |
| `prometheus` | `prom/prometheus:v2.53.0` | Collecte des indicateurs | 9090 |
| `grafana` | `grafana/grafana:11.1.0` | Restitution des indicateurs | 3000 |

**Choix d'architecture à savoir défendre.**

*Images épinglées à une version exacte, jamais `latest`.* Deux constructions du même
commit doivent produire la même image. `latest` rend ce résultat dépendant de la date.

*Réseau interne par nom de service.* L'API joint la base par `db:5432`, Prometheus joint
l'API par `api:8000`. Aucune adresse IP, aucun `localhost` entre conteneurs — `localhost`
dans un conteneur désigne ce conteneur, pas l'hôte.

*Le port publié concerne l'hôte.* `8000:8000` redirige le port de la machine vers celui du
conteneur ; les conteneurs, eux, communiquent sur le port interne.

*Un volume dissocie les données du conteneur.* `pgdata` conserve la base même si le
conteneur est détruit et recréé. Les artefacts MLflow vivent dans `mlruns`, distinct : les
métadonnées sont relationnelles, les artefacts sont des fichiers — deux natures, deux
stockages, conformément au § 3.

*Utilisateur non privilégié et image mince.* Le `Dockerfile` construit en deux étapes :
les outils d'installation restent dans l'étape jetable, l'image finale tourne sous
`appuser` et ne contient ni `pip` ni `setuptools`. Une compromission du service n'hérite
pas de root.

*Les secrets viennent d'un fichier `.env` non versionné*, et `gitleaks` s'exécute avant
chaque commit. Un secret committé reste dans l'historique même après suppression du
fichier : le seul contrôle efficace est celui qui l'empêche d'y entrer.

**Révision du chiffrage.** Les deux scénarios comparés plus haut restent valables pour le
seul service de scoring. La plateforme complète — base, registre, collecte, restitution —
ajoute un coût récurrent qui n'était pas dans le scénario A initial. Cet écart est assumé :
il achète la traçabilité exigée par C6 et C9, qu'un scoring batch nu ne fournit pas.


In [ ]:
# Section 11 - architecture diagram (to illustrate, e.g. diagram, outside executable code)


## 12. Mesure de performance et impacts (métriques techniques + métier) [C8]

**Performance technique (churn, classification) :** AUC, PR-AUC, matrice de confusion et rapport de
classification au seuil retenu (résultats section 9).

**Impact métier estimé.** Traduire la matrice de confusion en un ordre de grandeur d'impact business : nombre de
comptes correctement identifiés comme à risque (`TP`) × MRR moyen de ces comptes = revenu potentiellement
"sauvable" si l'action de rétention est efficace (à pondérer par un taux de succès hypothétique de la rétention,
à documenter comme hypothèse explicite).

**Régression `valeur_vie_client_eur` :** RMSE, MAE, R² sur un modèle de régression dédié (entraîné séparément,
sans réutiliser le churn comme feature explicative si l'on veut une lecture indépendante — à décider et justifier
selon l'objectif de cette cible secondaire).

**Journal de bord [C8] :**
- Décision : présenter systématiquement performance technique ET traduction métier côte à côte, pour rendre le
  modèle actionnable en soutenance devant un jury mixte technique/métier.

**Indicateurs, seuils d'alerte et actions déclenchées.**

Un indicateur sans seuil ne se surveille pas, et un seuil sans action associée ne sert à rien. Le tableau
ci-dessous ferme la boucle.

| Indicateur | Nature | Seuil d'alerte | Action déclenchée | Responsable |
|---|---|---|---|---|
| PR-AUC en production | Technique | Baisse > 15 % vs référence | Diagnostic, réentraînement | Équipe Data |
| Rappel au seuil retenu | Technique | < cible métier définie | Révision du seuil | Équipe Data + CSM |
| PSI sur les variables clés | Dérive | > 0,25 | Analyse de la dérive, réentraînement anticipé | Équipe Data |
| Taux de comptes signalés | Volumétrie | Écart > 30 % vs mois précédent | Contrôle qualité des données d'entrée | Équipe Data |
| Taux de manquants à l'entrée | Qualité | > 10 % sur une variable clé | Blocage du scoring, alerte | Pipeline (automatique) |
| Taux de rétention des comptes signalés | Métier | Non significatif vs comptes non signalés | Remise en cause de l'utilité de l'outil | Commanditaire |
| Retours de faux positifs par les CSM | Métier | Volume anormal | Révision du seuil ou des variables | CSM + Équipe Data |

Le dernier indicateur technique n'est pas le plus important : **un modèle performant dont les alertes ne
changent rien au taux de rétention ne crée aucune valeur.** L'indicateur métier est celui qui justifie le
maintien de la solution.

**Restitution aux interlocuteurs concernés.**

| Destinataire | Contenu | Fréquence |
|---|---|---|
| Équipes CSM | Liste priorisée des comptes à risque, score et principaux facteurs | Mensuelle |
| Commanditaire | Performance technique, impact métier estimé, incidents | Trimestrielle |
| Équipe technique | Dérive, qualité des données, incidents de pipeline | Mensuelle |

**Journal de bord [C8] — complément :**
- Décision : associer à chaque seuil une action et un responsable nommé. Motif : un tableau de bord sans
  destinataire d'action ne produit aucune décision.
- Décision : retenir le taux de rétention comparé entre comptes signalés et non signalés comme juge de paix.
  Motif : c'est le seul indicateur qui mesure l'utilité réelle, et non la qualité statistique du modèle.

### Indicateurs métier calibrés sur le portefeuille réel

Les indicateurs techniques ne disent rien de la valeur créée. Ceux-ci sont exprimés en euros de revenu
récurrent, seule unité que le commanditaire arbitre.

| Indicateur | Référence constatée | Lecture |
|---|---|---|
| MRR mensuel du portefeuille | ≈ 17,2 M€ | Assiette de référence |
| MRR porté par les comptes qui résilient | ≈ 4,0 M€ (23 %) | Exposition maximale théorique |
| MRR couvert par les 140 comptes traités | À mesurer | Mesure l'efficacité de la priorisation |
| Part du MRR à risque atteignable | ≈ 72 % avec les 10 % de churners les plus valorisés | Plafond de couverture à capacité constante |
| MRR effectivement préservé | Part du MRR couvert × efficacité de rétention observée | Valeur réellement créée |

**Trois niveaux à ne pas confondre**, et que le jury cherchera à distinguer :

1. Le MRR **exposé** — ce qui part si rien n'est fait.
2. Le MRR **couvert** — ce que la priorisation permet d'atteindre compte tenu de la capacité. Dépend du
   modèle et du classement.
3. Le MRR **préservé** — ce qui est réellement sauvé. Dépend en outre de l'efficacité des actions, qui ne
   relève pas du modèle.

Attribuer au modèle le troisième niveau serait une surestimation de son apport. Le modèle est responsable du
passage du niveau 1 au niveau 2 ; le passage au niveau 3 appartient aux équipes CSM.

**Mesure de référence indispensable.** Sans groupe de comparaison, l'effet réel est inobservable : les
comptes signalés étant par construction les plus à risque, comparer leur taux de rétention à la moyenne du
portefeuille produirait une conclusion fausse. Un échantillon témoin de comptes signalés mais non traités,
ou un déploiement progressif, est nécessaire pour établir l'effet causal. La variable
`groupe_experimentation` présente dans les données suggère qu'un tel dispositif existe déjà côté métier.

**Journal de bord [C8] — complément :**
- Décision : distinguer MRR exposé, couvert et préservé. Motif : confondre les trois conduit à surestimer
  l'apport du modèle d'un facteur égal à l'inverse de l'efficacité de rétention.
- Décision : exiger un groupe témoin avant toute revendication d'impact. Motif : sans lui, le biais de
  sélection garantit une mesure flatteuse et fausse.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- Business impact (example calculation, assumptions to document) ---
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
mrr_moyen_comptes_risque = df.loc[df.index.isin(X_test[y_pred == 1].index), "revenu_mensuel_recurrent_eur"].mean()
TAUX_SUCCES_RETENTION_HYPOTHESE = 0.25  # assumption to validate with the business
revenu_potentiellement_sauve = tp * mrr_moyen_comptes_risque * TAUX_SUCCES_RETENTION_HYPOTHESE
print(f"TP={tp} | MRR moyen comptes détectés à risque={mrr_moyen_comptes_risque:.1f} € "
      f"| Revenu mensuel potentiellement sauvé (hyp.)={revenu_potentiellement_sauve:.0f} €")

# --- valeur_vie_client_eur regression (separate model) ---
FEATURES_REGRESSION = [c for c in FEATURES if c != "churn"]  # do not reuse churn as a feature here
X_reg = df[FEATURES_REGRESSION]
y_reg = df["valeur_vie_client_eur"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

pipe_reg = Pipeline([
    ("prep", preprocess_tree),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42)),
]) if False else None  # placeholder: import RandomForestRegressor if selected

# from sklearn.ensemble import RandomForestRegressor
# pipe_reg = Pipeline([("prep", preprocess_tree), ("model", RandomForestRegressor(n_estimators=300, random_state=42))])
# pipe_reg.fit(Xr_train, yr_train)
# yr_pred = pipe_reg.predict(Xr_test)
# print("RMSE:", mean_squared_error(yr_test, yr_pred, squared=False))
# print("MAE:", mean_absolute_error(yr_test, yr_pred))
# print("R2:", r2_score(yr_test, yr_pred))


## 13. Amélioration continue (ré-entraînement, suivi, versioning) [C9]

**Dérive à surveiller.**
- Dérive des données (*data drift*) : distribution des variables d'usage évolue (ex. nouvelle fonctionnalité
  produit change `heures_usage_30j`).
- Dérive du concept (*concept drift*) : la relation usage → churn change (ex. nouvelle offre concurrente).

**Plan de ré-entraînement.** Ré-entraînement périodique (ex. trimestriel) ou déclenché par un seuil de dégradation
de l'AUC/PR-AUC observé sur les scores en production comparés aux churns réels constatés a posteriori.

**Monitoring proposé.**
- Suivi mensuel de la distribution des features clés vs distribution d'entraînement (ex. test de Kolmogorov-Smirnov
  ou PSI - Population Stability Index).
- Suivi de la performance réelle (AUC/PR-AUC) une fois les churns réels connus (délai d'observation = durée d'un
  cycle contractuel).

**Versioning.** Versionner le pipeline (code + hyperparamètres) et les données d'entraînement (ex. DVC ou simple
convention de nommage horodatée) pour garantir la traçabilité et le rollback en cas de régression.

**Journal de bord [C9] :**
- Décision : fixer une fréquence de ré-entraînement trimestrielle par défaut, ajustable si un monitoring de dérive
  déclenche une alerte avant cette échéance.

**Évaluation automatisée intégrée à la chaîne CI/CD.**

Le suivi décrit ci-dessus n'a de valeur que s'il est exécuté sans intervention humaine. Le dispositif
s'appuie sur la chaîne décrite en section 10 :

| Moment | Contrôle automatisé | Conséquence |
|---|---|---|
| Avant chaque scoring | Validation du schéma et de la qualité des données d'entrée | Blocage et alerte si échec |
| Après chaque scoring | Calcul du PSI vs distribution d'entraînement, journalisation | Alerte si dépassement de seuil |
| Après chaque réentraînement | Comparaison du candidat au modèle en production sur jeu de validation figé | Pas de promotion si dégradation |
| Mensuellement | Recalcul des métriques réelles dès que les issues sont connues | Alerte si dégradation |
| Trimestriellement | Revue de la pertinence des indicateurs et des seuils eux-mêmes | Ajustement documenté |

**Métriques intégrées au dispositif de suivi.**

| Catégorie | Métriques |
|---|---|
| Qualité de prévision | PR-AUC, AUC, rappel et précision au seuil retenu |
| Robustesse | Écart-type des scores en validation croisée, stabilité entre réentraînements |
| Variation de performance | Écart à la référence établie à la mise en production |
| Obsolescence | PSI par variable, ancienneté du modèle, ancienneté des données d'entraînement |
| Exploitation | Taux d'échec du pipeline, durée d'exécution, complétude des données d'entrée |

**Périodicité de révision des indicateurs.** Définie en phase de cadrage (section 2) : revue trimestrielle
des indicateurs et de leurs seuils, alignée sur le cycle de réentraînement. Cette revue ne porte pas sur la
performance du modèle mais sur **la pertinence des indicateurs eux-mêmes** — un seuil de PSI calibré à la
mise en service peut devenir inadapté après une évolution produit.

**Journal de bord [C9] — complément :**
- Décision : interdire la promotion automatique d'un modèle réentraîné. Motif : l'automatisation porte sur
  l'évaluation, pas sur la décision de mise en production — cohérent avec la supervision humaine posée en
  section 4.
- Décision : intégrer l'obsolescence des données d'entraînement comme métrique à part entière. Motif : un
  modèle peut rester performant sur des données anciennes tout en étant devenu inadapté au terrain.

### Boucle de rétroaction : monitoring → données

Le déclenchement d'une alerte ne conduit pas directement au réentraînement : il impose d'abord un retour à
l'étape **Données** du cycle de vie. Réentraîner sur un jeu de données dont la dérive n'a pas été comprise
reviendrait à apprendre le problème plutôt qu'à le corriger.

| Étape du retour | Contenu | Responsable |
|---|---|---|
| 1. Qualification de l'alerte | Dérive réelle, incident de collecte, ou évolution produit légitime ? | Équipe Data |
| 2. Nouvel instantané | Extraction fraîche, versionnée et figée selon la convention de la section 10 | Équipe Data |
| 3. Contrôle du schéma | Colonnes, types, domaines de valeurs, taux de manquants comparés aux références de la section 5 | Automatisé |
| 4. Comparaison des distributions | Nouvel instantané vs instantané d'entraînement en vigueur, variable par variable | Équipe Data |
| 5. Révision des features | Une variable dont la définition a changé côté produit doit être reconstruite, non réapprise | Équipe Data |
| 6. Mise à jour de la référence de dérive | La distribution de référence du PSI devient celle du nouvel instantané | Équipe Data |
| 7. Réentraînement | Uniquement après les étapes 1 à 6 | Équipe Data |

**Le point le plus souvent manqué est l'étape 6.** Si la distribution de référence servant au calcul du PSI
n'est pas mise à jour lors du réentraînement, le système continue de comparer les données courantes à un
état ancien : il signale indéfiniment une dérive déjà absorbée, ou cesse d'en détecter de nouvelles. La
référence de dérive est un artefact versionné au même titre que le modèle.

**Cas particulier — évolution produit.** Une modification fonctionnelle de la plateforme (nouveau plan,
refonte d'une fonctionnalité) produit une dérive légitime qui ne relève pas d'un défaut du modèle. Elle
impose néanmoins un réentraînement, car les variables d'usage changent de signification. Ce cas justifie que
l'équipe produit soit destinataire des alertes de dérive, au même titre que l'équipe Data.

**Journal de bord [C9] — complément :**
- Décision : interposer une étape de qualification entre l'alerte et le réentraînement. Motif : un incident
  de collecte et une dérive réelle produisent la même alerte mais appellent des réponses opposées —
  correction du pipeline dans un cas, réapprentissage dans l'autre.
- Décision : versionner la distribution de référence du PSI comme un artefact à part entière. Motif : sans
  mise à jour, le dispositif de détection de dérive devient inopérant après le premier réentraînement.

### Mise en œuvre de la surveillance

Les indicateurs du § 12 sont publiés par un exporteur Prometheus (`monitoring/exporteur.py`)
et collectés toutes les 30 secondes. Grafana est provisionné au démarrage : sa source de
données est configurée par fichier, sans intervention manuelle — le tableau de bord est
donc reproductible.

| Indicateur exposé | Nature | Origine |
|---|---|---|
| `churn_comptes_scores` | volumétrie | lot mensuel |
| `churn_comptes_a_traiter` | volumétrie | lot mensuel |
| `churn_valeur_esperee_totale_eur` | métier | lot mensuel |
| `churn_psi_max` | dérive | calcul de dérive |
| `churn_pr_auc` | technique | dernière évaluation |
| `churn_taux_manquants_max` | qualité | contrôle d'entrée |
| latence, codes de retour | technique | instrumentation de l'API |

**Une séparation de responsabilité à souligner.** L'exporteur **expose**, il ne décide pas.
Les seuils, les actions associées et les responsables nommés restent dans
`monitoring/alertes.py`, en Python testé. Coder un seuil dans la configuration Prometheus
le rendrait invisible depuis le code et difficile à tester ; le laisser dans le code le
rend vérifiable par la suite de tests et lisible par le jury.


In [ ]:
# Section 13 - no executable code here (operational plan), cell reserved for a monitoring script


## 14. Conclusion (synthèse et recommandations)

*(À rédiger après exécution complète du notebook.)*

- Rappel de la performance retenue (AUC/PR-AUC, seuil, impact métier estimé).
- Rappel des limites (variables leurres confirmées, hypothèses de coût métier, taille et représentativité du jeu
  de données synthétique).
- Recommandations opérationnelles pour les équipes Customer Success (ex. priorisation des comptes à forte MRR et
  score de risque élevé).
- Prochaines étapes (cf. section 13 : monitoring, ré-entraînement, enrichissement des données).


## 15. Annexes (versions, paramètres, dépendances, fonctions utilitaires)

- Versions des bibliothèques utilisées (à générer via `pip freeze` ou `%watermark`).
- Hyperparamètres complets des modèles entraînés (`MODELE_FINAL.get_params()`).
- Fonctions utilitaires définies dans le notebook (`nettoyer_decimal_texte`, `parser_dates_multiformat`).
- Dictionnaire de données complet (`DATA_DICT`, section 5).

---

### Annexe A — Reproductibilité

*(voir la cellule de code ci-dessous : versions, dépendances, paramètres.)*

---

### Annexe B — Traçabilité : critères du référentiel ↔ emplacement de la preuve

Cette table permet de retrouver directement l'élément démontrant chaque critère évalué.

| Compétence | Critère | Section |
|---|---|---|
| **C1** | Besoins métiers identifiés, cas d'usage décrits | 2 |
| **C1** | Données pertinentes identifiées | 3, 5 |
| **C1** | Existence, disponibilité et accès vérifiés | 3 |
| **C1** | Solutions alternatives en cas d'indisponibilité | 3 |
| **C2** | Cadres éthiques connus et appliqués | 4 |
| **C2** | Impacts et conséquences du déploiement | 4 |
| **C2** | Biais potentiels identifiés | 4, 6 |
| **C2** | Dilemmes éthiques identifiés | 4 |
| **C2** | Risques portés à la connaissance des acteurs | 4 |
| **C2** | Vérification légale du jeu de données | 4 |
| **C3** | Nommage et formats adaptés | 5, 7 |
| **C3** | Données altérées, doublons et blancs traités | 7 |
| **C3** | Traitements documentés | 7 |
| **C3** | Modèle de stockage justifié | 3, 11 |
| **C3** | Cycle de vie du jeu de données documenté | 3 |
| **C3** | Cycle de vie soumis aux parties prenantes | 3, 4 |
| **C4** | Pertinence évaluée par analyse ROC | 8, 12 |
| **C4** | Contraintes opérationnelles prises en compte | 8, 11 |
| **C4** | Contraintes d'éco-conception portées aux acteurs | 8, 11 |
| **C4** | Familles d'algorithmes connues | 8 |
| **C4** | Démarche scientifique documentée | 8 |
| **C4** | Performance attendue déterminée | 8 |
| **C4** | Type de résultat identifié | 8 |
| **C4** | Solutions sur étagère évaluées | 8 |
| **C5** | Modèle optimisé selon le contexte | 9 |
| **C5** | Hyperparamètres décrits | 9, 15 |
| **C5** | Feature engineering effectué | 7 |
| **C5** | Réentraînement prévu | 13 |
| **C6** | Livraison et déploiement continus | 10 |
| **C6** | Versioning implémenté | 10 |
| **C6** | Besoins d'intégration documentés | 10 |
| **C7** | Architectures et contraintes connues | 11 |
| **C7** | Contraintes économiques chiffrées et portées aux acteurs | 11 |
| **C7** | Acteurs interrogés sur les contraintes | 11 |
| **C8** | Indicateurs et seuils associés définis | 12 |
| **C8** | Performance mesurée par suivi des indicateurs | 12, 13 |
| **C8** | Résultats interprétés et présentés | 12 |
| **C8** | Actions déclenchées selon les résultats | 12 |
| **C9** | Évaluation automatisée intégrée au CI/CD | 13 |
| **C9** | Métriques de robustesse et d'obsolescence intégrées | 13 |
| **C9** | Périodicité de révision définie en cadrage | 2, 13 |

---

### Annexe C — Correspondance notebook ↔ soutenance orale (30 minutes)

La soutenance ne rejoue pas le notebook : elle en restitue le raisonnement. Cette table sert de fil
conducteur au support de présentation.

| Temps | Propos | Sections mobilisées |
|---|---|---|
| 3 min | Présentation du candidat et du problème métier | 2 |
| 3 min | Cadrage : besoin, valeur attendue, ce que le modèle n'est pas | 2 |
| 4 min | Données : sources, qualité constatée, gouvernance, cycle de vie | 3, 5 |
| 3 min | Éthique : biais, cadres applicables, dilemmes, variables exclues | 4 |
| 4 min | Démarche : enseignements de l'EDA, fuite détectée, préparation | 6, 7 |
| 5 min | Modélisation : baseline, modèle retenu, validation, seuil et coût métier | 8, 9 |
| 4 min | Résultats : métriques et traduction métier | 12 |
| 3 min | Exploitation : architecture, intégration, suivi | 10, 11, 13 |
| 3 min | Limites assumées, pistes, recommandation | 14, 13 |

Points que le jury interroge le plus souvent, et section où se trouve la réponse : le choix du seuil (9), la
détection de la fuite (7), l'arbitrage entre modèles (8), la vérification des biais (4), la conduite à tenir
en cas de dégradation en production (13), la valeur métier chiffrée et ses hypothèses (12).

---

### Annexe D — Journal des itérations

Récapitulatif des retours en arrière effectués au cours du projet. Cette table matérialise le caractère
itératif de la démarche : chaque ligne correspond à une boucle du cycle de vie parcourue en sens inverse.

| # | Étape atteinte | Constat déclencheur | Retour vers | Modification apportée | Effet mesuré |
|---|---|---|---|---|---|
| 1 | Modélisation | Performance anormalement élevée du premier modèle | **Cadrage** | Formalisation de la frontière temporelle avant/après décision ; exclusion de `sante_compte_fin_periode` | Performance ramenée à un niveau plausible |
| 2 | Analyse métier | 10 % des churners portent 72 % du MRR perdu | **Cadrage** | Objectif reformulé : priorisation par valeur au lieu de rappel global ; capacité CSM ajoutée aux contraintes | Règle de décision revue (§ 9), conséquence éthique traitée (§ 4) |
| 3 | Évaluation | *(à compléter : apport mesuré du feature engineering)* | **Features** | *(à compléter)* | *(à compléter)* |
| 4 | *(à compléter si une itération supplémentaire survient)* | | | | |

**Lecture.** Les itérations 1 et 2 sont des retours au cadrage : elles ont modifié la définition du problème,
pas seulement sa résolution. L'itération 3 est un retour à la construction des variables, de portée plus
locale. Les boucles *monitoring → données* (§ 13) relèvent de l'exploitation et ne figurent pas ici : elles
n'ont pas encore été parcourues, le modèle n'étant pas déployé.

**Portée méthodologique.** Une démarche scientifique se reconnaît moins aux hypothèses confirmées qu'aux
hypothèses invalidées. Les deux retours au cadrage documentés ci-dessus ont l'un et l'autre remis en cause un
présupposé initial : la disponibilité des variables d'une part, l'objectif de détection exhaustive d'autre
part.


---

### Annexe E — Organisation du code : un seul code, un notebook auto-porteur

**La tension à résoudre.** Le règlement exige un notebook compréhensible sans explication
orale : le lecteur doit voir le code. L'ingénierie exige un code écrit et testé une seule
fois : du code dupliqué diverge toujours. Recopier les fonctions entre le notebook et un
module produirait deux versions, dont une seule serait testée.

**La solution retenue.** Une source de vérité unique — le paquet `src/churn_saas/`, testé
par `pytest` — et un notebook qui **affiche le code au moment où il l'explique**, en le
lisant dans le module :

```python
from churn_saas.notebook import preparer_import, afficher_source
preparer_import()   # rend churn_saas importable, que le paquet soit installé ou non

from churn_saas.donnees.silver import nettoyer_decimal_texte
afficher_source(nettoyer_decimal_texte)          # le jury lit le code ici même
df["taux_adoption_pct"] = nettoyer_decimal_texte(df["taux_adoption_pct"])
```

Le code affiché **est** le code exécuté : il est lu dans le module au moment de
l'affichage et ne peut donc pas diverger de la version testée. Un test le vérifie
(`tests/test_notebook.py`). Le notebook reste auto-porteur ; le code n'existe qu'à un
seul endroit.

**Découpage par activité.** Un dossier par étape du cycle de vie, un fichier de tests par
dossier.

| Activité | Module | Tests | Sections |
|---|---|---|---|
| 1 · Gestion des données (bronze → silver → gold) | `donnees/` | `test_donnees.py` | 3, 5, 7 |
| 2 · Contrôle des features | `features/` | `test_features.py` | 6, 7 |
| 3 · Modélisation (baseline, sélection) | `modelisation/` | — | 8, 9 |
| 4 · Évaluation (métriques, décision, impact) | `evaluation/` | `test_evaluation.py` | 9, 12 |
| 5 · Packaging (artefacts, fiche modèle) | `packaging/` | `test_packaging.py` | 10 |
| 6 · Industrialisation (lot mensuel, service) | `industrialisation/` | — | 10, 11 |
| 7 · Monitoring (dérive, alertes) | `monitoring/` | `test_monitoring.py` | 12, 13 |

**Deux frontières portent une décision, pas une commodité de rangement.**

*silver → gold* : c'est au passage au gold que les colonnes interdites sont retirées. La
règle est générale — aucune variable postérieure à la décision — et non une liste de noms.
`donnees/gold.py` porte la table des motifs, distincts et non interchangeables : fuite
temporelle, identifiant, RGPD, artefact de process, cible secondaire.

*lot mensuel ≠ service unitaire* : `industrialisation/scoring.py` décide, parce qu'il voit
tout le portefeuille et peut classer. `industrialisation/service.py` ne décide pas, parce
qu'il voit un compte à la fois. Le service ne renvoie donc aucun champ `decision` — cette
absence est volontaire.

**Ce que les tests protègent** : ce qui casse sans bruit. Une jointure `STARTER`/`starter`
qui perd des lignes sans lever d'erreur ; une virgule décimale mal convertie ; la
réintroduction accidentelle d'une variable en fuite ; une division par zéro qui propage un
infini ; la perte de l'argument de robustesse du § 9.

Le détail figure dans `docs/ORGANISATION_CODE.md`, et la chaîne CI du § 10 exécute ces
tests à chaque modification.


In [ ]:
import sys
print("Python:", sys.version)
for pkg in ["pandas", "numpy", "sklearn", "matplotlib", "seaborn"]:
    mod = __import__(pkg)
    print(pkg, getattr(mod, "__version__", "n/a"))

print("\nHyperparamètres du modèle final :")
print(MODELE_FINAL.get_params())
